# NEURODYNAFUSION — part 1 of 1

This notebook was automatically split from a larger notebook.

### Tensor Creation

In [7]:
import os
import re
import io
import json
import glob
import zipfile
import hashlib
import warnings
import time
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

MODE = "add_patient"  # options: "add_patient", "build_training_bundle"

PATIENT_ID = "sub001"
LABEL = "control"
SITE = "siteA"
RUN_ID = "run1"
PATIENT_DYNAMICAL_RESULTS_DIR = "/home/a/projects/Complete-Neural-Signal-Analysis/results"

SPEED_MODE = "ultra"  # options: "ultra", "fast", "balanced", "full"

USE_EXISTING_PATIENT_TENSOR_IF_PRESENT = False
PRINT_EVERY_N_FILES = 10

dataset_bank_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_tensor_bank")
patient_tensor_dir = os.path.join(dataset_bank_dir, "patient_tensors")
source_report_dir = os.path.join(dataset_bank_dir, "source_reports")
os.makedirs(dataset_bank_dir, exist_ok=True)
os.makedirs(patient_tensor_dir, exist_ok=True)
os.makedirs(source_report_dir, exist_ok=True)

patient_index_csv = os.path.join(dataset_bank_dir, "patient_tensor_index.csv")
training_bundle_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_bundle.pt")
training_metadata_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_metadata.json")
training_manifest_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_sample_manifest.csv")
modality_registry_path = os.path.join(dataset_bank_dir, "modality_registry.json")

if SPEED_MODE == "ultra":
    dynamic_image_h = 32
    dynamic_image_w = 32
    dynamic_token_features = 32
    max_file_bytes = 80_000_000
    max_arrays_per_file = 8
    max_values_per_array = 120_000
    max_zip_members = 0
    max_files_to_scan = 350
    include_images = False
    include_text_like = False
    include_zip = False
    save_float16_for_images = True
elif SPEED_MODE == "fast":
    dynamic_image_h = 48
    dynamic_image_w = 48
    dynamic_token_features = 40
    max_file_bytes = 150_000_000
    max_arrays_per_file = 16
    max_values_per_array = 300_000
    max_zip_members = 200
    max_files_to_scan = 800
    include_images = True
    include_text_like = False
    include_zip = True
    save_float16_for_images = True
elif SPEED_MODE == "balanced":
    dynamic_image_h = 64
    dynamic_image_w = 64
    dynamic_token_features = 48
    max_file_bytes = 300_000_000
    max_arrays_per_file = 32
    max_values_per_array = 750_000
    max_zip_members = 1200
    max_files_to_scan = 2000
    include_images = True
    include_text_like = True
    include_zip = True
    save_float16_for_images = True
elif SPEED_MODE == "full":
    dynamic_image_h = 64
    dynamic_image_w = 64
    dynamic_token_features = 48
    max_file_bytes = 500_000_000
    max_arrays_per_file = 64
    max_values_per_array = 1_500_000
    max_zip_members = 5000
    max_files_to_scan = None
    include_images = True
    include_text_like = True
    include_zip = True
    save_float16_for_images = True
else:
    raise ValueError("SPEED_MODE must be one of: 'ultra', 'fast', 'balanced', 'full'.")

base_extensions = {".csv", ".npz", ".npy", ".json"}
image_extensions = {".png", ".jpg", ".jpeg"}
text_extensions = {".txt", ".md", ".log", ".html"}

allowed_extensions = set(base_extensions)
if include_images:
    allowed_extensions |= image_extensions
if include_text_like:
    allowed_extensions |= text_extensions
if include_zip:
    allowed_extensions |= {".zip"}

exclude_path_markers = [
    "/__pycache__/",
    "/patient_level_all_dynamical_tensor_bank/",
    "/patient_level_dynamical_tensor_bank/",
    "/neurodynafusion_disease_tensors/",
    "/neurodynafusion_disease_tensors_numpy/",
    "/neurodynafusion_disease_tensors_all_dynamical/",
    "/patient_level_all_dynamical_neural_net/",
    "/expert_neural_net_plots/",
]

eeg_channel_names = [
    "Fp1", "Fpz", "Fp2", "F7", "F3", "Fz", "F4", "F8", "FC5", "FC1", "FC2", "FC6",
    "M1", "T7", "C3", "Cz", "C4", "T8", "M2", "CP5", "CP1", "CP2", "CP6",
    "P7", "P3", "Pz", "P4", "P8", "POz", "O1", "Oz", "O2"
]

eps = 1e-12

def log_msg(msg):
    print(msg, flush=True)

def safe_filename(x):
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x

def safe_nan_to_num(x):
    x = np.asarray(x, dtype=float)
    if np.any(~np.isfinite(x)):
        finite = x[np.isfinite(x)]
        med = np.nanmedian(finite) if finite.size else 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x

def robust_vector_zscore(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        return np.zeros_like(x, dtype=np.float32)
    med = np.nanmedian(finite)
    q25 = np.nanpercentile(finite, 25)
    q75 = np.nanpercentile(finite, 75)
    iqr = q75 - q25
    x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    if not np.isfinite(iqr) or iqr <= eps:
        mu = np.mean(x)
        sd = np.std(x)
        return ((x - mu) / (sd + eps)).astype(np.float32)
    return ((x - med) / (iqr + eps)).astype(np.float32)

def numeric_array_or_none(x):
    try:
        arr = np.asarray(x)
    except Exception:
        return None
    if arr.dtype == object:
        try:
            if arr.shape == ():
                obj = arr.item()
                if isinstance(obj, dict):
                    return None
            arr = arr.astype(float)
        except Exception:
            return None
    if not np.issubdtype(arr.dtype, np.number):
        return None
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0:
        return None
    if arr.size > max_values_per_array:
        idx = np.linspace(0, arr.size - 1, max_values_per_array).astype(int)
        arr = arr.reshape(-1)[idx]
    return arr

def collect_numeric_arrays_from_object(obj, arrays, depth=0):
    if len(arrays) >= max_arrays_per_file or depth > 5:
        return
    arr = numeric_array_or_none(obj)
    if arr is not None and arr.size >= 1:
        arrays.append(arr)
        return
    if isinstance(obj, np.ndarray) and obj.dtype == object and obj.shape == ():
        try:
            collect_numeric_arrays_from_object(obj.item(), arrays, depth + 1)
        except Exception:
            return
    elif isinstance(obj, dict):
        for _, v in obj.items():
            collect_numeric_arrays_from_object(v, arrays, depth + 1)
            if len(arrays) >= max_arrays_per_file:
                break
    elif isinstance(obj, (list, tuple)):
        arr = numeric_array_or_none(obj)
        if arr is not None and arr.size >= 1:
            arrays.append(arr)
            return
        for v in obj[:64]:
            collect_numeric_arrays_from_object(v, arrays, depth + 1)
            if len(arrays) >= max_arrays_per_file:
                break

def is_excluded_path(path):
    p = str(path).replace("\\", "/")
    for marker in exclude_path_markers:
        if marker in p:
            return True
    return False

def find_patient_result_files(patient_dir):
    if not os.path.isdir(patient_dir):
        raise FileNotFoundError(f"Missing patient dynamical results directory: {patient_dir}")

    log_msg(f"Scanning folder: {patient_dir}")
    log_msg(f"SPEED_MODE: {SPEED_MODE}")
    log_msg(f"Allowed extensions: {sorted(allowed_extensions)}")

    paths = []
    for ext in allowed_extensions:
        paths.extend(glob.glob(os.path.join(patient_dir, "**", f"*{ext}"), recursive=True))

    clean = []
    skipped_size = 0
    skipped_excluded = 0

    for p in sorted(set(paths)):
        if not os.path.isfile(p):
            continue
        if is_excluded_path(p):
            skipped_excluded += 1
            continue
        try:
            if os.path.getsize(p) > max_file_bytes:
                skipped_size += 1
                continue
        except Exception:
            continue
        clean.append(p)

    clean = sorted(clean, key=lambda x: os.path.getsize(x) if os.path.exists(x) else 0)

    if max_files_to_scan is not None and len(clean) > max_files_to_scan:
        log_msg(f"Limiting scan from {len(clean)} files to {max_files_to_scan} smallest files for {SPEED_MODE} mode.")
        clean = clean[:max_files_to_scan]

    log_msg(f"Candidate files: {len(clean)}")
    log_msg(f"Skipped by size: {skipped_size}")
    log_msg(f"Skipped excluded/generated folders: {skipped_excluded}")

    return clean

def load_numeric_arrays_from_csv_bytes(b):
    arrays = []
    try:
        df = pd.read_csv(io.BytesIO(b))
    except Exception:
        return arrays
    if df.empty:
        return arrays
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_cols:
        return arrays
    channel_cols = [c for c in df.columns if str(c).lower() == "channel"]
    if channel_cols:
        channel_col = channel_cols[0]
        rows = []
        for ch in eeg_channel_names:
            sub = df[df[channel_col].astype(str) == ch]
            if len(sub) == 0:
                rows.append(np.full(len(numeric_cols), np.nan))
            else:
                rows.append(sub[numeric_cols].mean(numeric_only=True).to_numpy(dtype=float))
        arrays.append(np.vstack(rows))
    else:
        arr = df[numeric_cols].to_numpy(dtype=float)
        if arr.size > max_values_per_array:
            flat = arr.reshape(-1)
            idx = np.linspace(0, flat.size - 1, max_values_per_array).astype(int)
            arr = flat[idx]
        arrays.append(arr)
    return arrays

def load_numeric_arrays_from_csv(path):
    with open(path, "rb") as f:
        return load_numeric_arrays_from_csv_bytes(f.read())

def load_numeric_arrays_from_npz_bytes(b):
    arrays = []
    try:
        z = np.load(io.BytesIO(b), allow_pickle=True)
    except Exception:
        return arrays
    keys = list(z.keys())[:max_arrays_per_file]
    for k in keys:
        collect_numeric_arrays_from_object(z[k], arrays)
        if len(arrays) >= max_arrays_per_file:
            break
    return arrays

def load_numeric_arrays_from_npz(path):
    with open(path, "rb") as f:
        return load_numeric_arrays_from_npz_bytes(f.read())

def load_numeric_arrays_from_npy_bytes(b):
    arrays = []
    try:
        obj = np.load(io.BytesIO(b), allow_pickle=True)
    except Exception:
        return arrays
    collect_numeric_arrays_from_object(obj, arrays)
    return arrays

def load_numeric_arrays_from_npy(path):
    with open(path, "rb") as f:
        return load_numeric_arrays_from_npy_bytes(f.read())

def load_numeric_arrays_from_json_bytes(b):
    arrays = []
    try:
        obj = json.loads(b.decode("utf-8", errors="ignore"))
    except Exception:
        return arrays
    collect_numeric_arrays_from_object(obj, arrays)
    return arrays

def load_numeric_arrays_from_json(path):
    with open(path, "rb") as f:
        return load_numeric_arrays_from_json_bytes(f.read())

def load_numeric_arrays_from_text_bytes(b):
    if not include_text_like:
        return []
    text = b.decode("utf-8", errors="ignore")
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?", text)
    if len(nums) < 2:
        return []
    nums = nums[:max_values_per_array]
    vals = np.asarray([float(x) for x in nums], dtype=float)
    return [vals]

def load_numeric_arrays_from_text(path):
    with open(path, "rb") as f:
        return load_numeric_arrays_from_text_bytes(f.read())

def load_image_array_from_bytes(b):
    if not include_images:
        return []
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(b)).convert("L")
        img.thumbnail((max(dynamic_image_w * 2, 64), max(dynamic_image_h * 2, 64)))
        return [np.asarray(img, dtype=float)]
    except Exception:
        return []

def load_image_array_from_path(path):
    with open(path, "rb") as f:
        return load_image_array_from_bytes(f.read())

def load_arrays_from_bytes_by_ext(b, ext):
    ext = ext.lower()
    if ext == ".csv":
        return load_numeric_arrays_from_csv_bytes(b), "numeric"
    if ext == ".npz":
        return load_numeric_arrays_from_npz_bytes(b), "numeric"
    if ext == ".npy":
        return load_numeric_arrays_from_npy_bytes(b), "numeric"
    if ext == ".json":
        return load_numeric_arrays_from_json_bytes(b), "numeric"
    if ext in text_extensions:
        return load_numeric_arrays_from_text_bytes(b), "text_numeric"
    if ext in image_extensions:
        return load_image_array_from_bytes(b), "image"
    return [], "unsupported"

def load_arrays_from_path(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return load_numeric_arrays_from_csv(path), "numeric"
    if ext == ".npz":
        return load_numeric_arrays_from_npz(path), "numeric"
    if ext == ".npy":
        return load_numeric_arrays_from_npy(path), "numeric"
    if ext == ".json":
        return load_numeric_arrays_from_json(path), "numeric"
    if ext in text_extensions:
        return load_numeric_arrays_from_text(path), "text_numeric"
    if ext in image_extensions:
        return load_image_array_from_path(path), "image"
    return [], "unsupported"

def resize_2d_nearest(x, h, w):
    x = np.asarray(x, dtype=float)
    if x.ndim == 0:
        x = x.reshape(1, 1)
    elif x.ndim == 1:
        x = x.reshape(-1, 1)
    elif x.ndim > 2:
        channel_axis = None
        for ax, size in enumerate(x.shape):
            if size == len(eeg_channel_names):
                channel_axis = ax
                break
        if channel_axis is not None:
            x = np.moveaxis(x, channel_axis, 0).reshape(x.shape[channel_axis], -1)
        else:
            x = x.reshape(x.shape[0], -1)
    x = safe_nan_to_num(x)
    if x.shape[0] == 0 or x.shape[1] == 0:
        return np.zeros((h, w), dtype=np.float32)
    row_idx = np.linspace(0, x.shape[0] - 1, h).astype(int)
    col_idx = np.linspace(0, x.shape[1] - 1, w).astype(int)
    return x[np.ix_(row_idx, col_idx)].astype(np.float32)

def array_to_image2d(arr, h, w):
    arr = numeric_array_or_none(arr)
    if arr is None:
        return np.zeros((h, w), dtype=np.float32)
    arr = np.squeeze(arr)
    if arr.ndim == 0:
        arr = arr.reshape(1, 1)
    elif arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    elif arr.ndim > 2:
        channel_axis = None
        for ax, size in enumerate(arr.shape):
            if size == len(eeg_channel_names):
                channel_axis = ax
                break
        if channel_axis is not None:
            arr = np.moveaxis(arr, channel_axis, 0)
            arr = arr.reshape(arr.shape[0], -1)
        else:
            arr = arr.reshape(arr.shape[0], -1)
    img = resize_2d_nearest(arr, h, w)
    img = robust_vector_zscore(img).reshape(h, w)
    img = np.clip(img, -8.0, 8.0).astype(np.float32)
    return img

def arrays_to_modality_image(arrays, h, w):
    arrays = [a for a in arrays if numeric_array_or_none(a) is not None]
    if len(arrays) == 0:
        return None
    arrays = arrays[:max_arrays_per_file]
    if len(arrays) == 1:
        return array_to_image2d(arrays[0], h, w)
    mini_h = max(4, h // min(len(arrays), 6))
    pieces = [array_to_image2d(arr, mini_h, w) for arr in arrays]
    stacked = np.vstack(pieces)
    img = resize_2d_nearest(stacked, h, w)
    img = robust_vector_zscore(img).reshape(h, w)
    img = np.clip(img, -8.0, 8.0).astype(np.float32)
    return img

def array_stats(arr):
    arr = numeric_array_or_none(arr)
    if arr is None:
        return np.zeros(dynamic_token_features, dtype=np.float32)
    x = np.asarray(arr, dtype=float).reshape(-1)
    if x.size > max_values_per_array:
        idx = np.linspace(0, x.size - 1, max_values_per_array).astype(int)
        x = x[idx]
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        return np.zeros(dynamic_token_features, dtype=np.float32)
    q = np.percentile(finite, [1, 5, 10, 25, 50, 75, 90, 95, 99])
    mu = np.mean(finite)
    sd = np.std(finite)
    mn = np.min(finite)
    mx = np.max(finite)
    rms = np.sqrt(np.mean(finite ** 2))
    abs_mean = np.mean(np.abs(finite))
    pos_frac = np.mean(finite > 0)
    neg_frac = np.mean(finite < 0)
    zero_frac = np.mean(finite == 0)
    nan_frac = 1.0 - finite.size / max(x.size, 1)
    centered = finite - mu
    skew = np.mean(centered ** 3) / ((sd + eps) ** 3)
    kurt = np.mean(centered ** 4) / ((sd + eps) ** 4)
    energy = np.sum(finite ** 2) / finite.size
    diff = np.diff(finite)
    stats = np.array([
        mu, sd, mn, mx, rms, abs_mean,
        q[0], q[1], q[2], q[3], q[4], q[5], q[6], q[7], q[8],
        pos_frac, neg_frac, zero_frac, nan_frac,
        skew, kurt, energy, np.log10(abs(energy) + eps),
        np.log10(finite.size + 1), np.log10(np.asarray(arr).size + 1), np.asarray(arr).ndim,
        finite[0], finite[-1],
        np.mean(diff) if diff.size else 0.0,
        np.std(diff) if diff.size else 0.0,
        mx - mn, q[7] - q[1],
        np.median(finite),
        np.mean(np.abs(diff)) if diff.size else 0.0,
        np.percentile(np.abs(finite), 95),
        np.percentile(np.abs(finite), 99),
        np.mean(finite > np.percentile(finite, 75)),
        np.mean(finite < np.percentile(finite, 25)),
        np.log10(abs(mu) + eps),
        np.log10(sd + eps),
        np.log10(abs(mx - mn) + eps),
        np.sign(mu),
        np.sign(finite[0]),
        np.sign(finite[-1]),
        finite.size / max(x.size, 1),
        np.isfinite(np.asarray(arr, dtype=float)).mean(),
        np.asarray(arr).ndim,
        np.log10(np.asarray(arr).shape[0] + 1) if np.asarray(arr).ndim >= 1 else 0.0,
    ], dtype=float)
    if stats.size < dynamic_token_features:
        stats = np.pad(stats, (0, dynamic_token_features - stats.size))
    elif stats.size > dynamic_token_features:
        stats = stats[:dynamic_token_features]
    stats = robust_vector_zscore(stats)
    stats = np.nan_to_num(stats, nan=0.0, posinf=0.0, neginf=0.0)
    return stats.astype(np.float32)

def arrays_to_token(arrays):
    arrays = [a for a in arrays if numeric_array_or_none(a) is not None]
    if len(arrays) == 0:
        return np.zeros(dynamic_token_features, dtype=np.float32)
    stats = np.vstack([array_stats(a) for a in arrays[:max_arrays_per_file]])
    token = np.mean(stats, axis=0)
    token = np.nan_to_num(token, nan=0.0, posinf=0.0, neginf=0.0)
    return token.astype(np.float32)

def modality_name_from_path(path, patient_dir, zip_member=None):
    if zip_member is None:
        rel = os.path.relpath(path, patient_dir)
    else:
        rel = os.path.relpath(path, patient_dir) + "__" + zip_member
    rel = rel.replace("\\", "/")
    no_ext = os.path.splitext(rel)[0]
    name = re.sub(r"[^A-Za-z0-9_]+", "_", no_ext)
    name = re.sub(r"_+", "_", name).strip("_")
    h = hashlib.sha1(rel.encode("utf-8")).hexdigest()[:8]
    return f"{name}_{h}"

def build_records_from_regular_file(path, patient_dir):
    arrays, source_kind = load_arrays_from_path(path)
    modality_name = modality_name_from_path(path, patient_dir)
    return [{
        "modality_name": modality_name,
        "path": path,
        "zip_member": "",
        "source_kind": source_kind,
        "arrays": arrays,
        "file_size_bytes": os.path.getsize(path) if os.path.exists(path) else 0,
    }]

def build_records_from_zip(path, patient_dir):
    records = []
    if not include_zip or max_zip_members <= 0:
        return records
    try:
        with zipfile.ZipFile(path, "r") as zf:
            infos = [i for i in zf.infolist() if not i.is_dir()]
            infos = infos[:max_zip_members]
            for info_i, info in enumerate(infos):
                member = info.filename
                ext = os.path.splitext(member)[1].lower()
                if ext not in allowed_extensions or ext == ".zip":
                    continue
                try:
                    b = zf.read(info)
                except Exception:
                    continue
                arrays, source_kind = load_arrays_from_bytes_by_ext(b, ext)
                records.append({
                    "modality_name": modality_name_from_path(path, patient_dir, member),
                    "path": path,
                    "zip_member": member,
                    "source_kind": f"zip_{source_kind}",
                    "arrays": arrays,
                    "file_size_bytes": len(b),
                })
    except Exception:
        pass
    return records

def build_patient_dynamical_tensors(patient_dir):
    t0 = time.time()
    files = find_patient_result_files(patient_dir)

    images = []
    tokens = []
    modality_names = []
    report_rows = []

    loaded_count = 0
    skipped_count = 0

    log_msg("Beginning file processing...")

    for file_i, p in enumerate(files, start=1):
        if file_i == 1 or file_i % PRINT_EVERY_N_FILES == 0 or file_i == len(files):
            elapsed = time.time() - t0
            log_msg(f"[{file_i}/{len(files)}] loaded={loaded_count}, skipped={skipped_count}, elapsed={elapsed:.1f}s | {os.path.basename(p)}")

        ext = os.path.splitext(p)[1].lower()

        if ext == ".zip":
            records = build_records_from_zip(p, patient_dir)
            if len(records) == 0:
                report_rows.append({
                    "patient_id": PATIENT_ID,
                    "run_id": RUN_ID,
                    "modality_index": -1,
                    "modality_name": modality_name_from_path(p, patient_dir),
                    "path": p,
                    "zip_member": "",
                    "source_kind": "zip_skipped",
                    "status": "skipped_zip_disabled_or_empty",
                    "n_arrays": 0,
                    "array_shapes_first": "",
                    "file_size_bytes": os.path.getsize(p) if os.path.exists(p) else 0,
                })
                skipped_count += 1
                continue
        else:
            records = build_records_from_regular_file(p, patient_dir)

        for rec in records:
            arrays = [a for a in rec["arrays"] if numeric_array_or_none(a) is not None]

            if len(arrays) == 0:
                report_rows.append({
                    "patient_id": PATIENT_ID,
                    "run_id": RUN_ID,
                    "modality_index": -1,
                    "modality_name": rec["modality_name"],
                    "path": rec["path"],
                    "zip_member": rec["zip_member"],
                    "source_kind": rec["source_kind"],
                    "status": "skipped_no_numeric_or_image_arrays",
                    "n_arrays": 0,
                    "array_shapes_first": "",
                    "file_size_bytes": rec["file_size_bytes"],
                })
                skipped_count += 1
                continue

            img = arrays_to_modality_image(arrays, dynamic_image_h, dynamic_image_w)
            tok = arrays_to_token(arrays)

            if img is None:
                report_rows.append({
                    "patient_id": PATIENT_ID,
                    "run_id": RUN_ID,
                    "modality_index": -1,
                    "modality_name": rec["modality_name"],
                    "path": rec["path"],
                    "zip_member": rec["zip_member"],
                    "source_kind": rec["source_kind"],
                    "status": "skipped_image_failed",
                    "n_arrays": len(arrays),
                    "array_shapes_first": "",
                    "file_size_bytes": rec["file_size_bytes"],
                })
                skipped_count += 1
                continue

            idx = len(images)
            images.append(img.astype(np.float32))
            tokens.append(tok.astype(np.float32))
            modality_names.append(rec["modality_name"])
            loaded_count += 1

            shapes = []
            for a in arrays[:8]:
                try:
                    shapes.append(str(tuple(np.asarray(a).shape)))
                except Exception:
                    shapes.append("?")

            report_rows.append({
                "patient_id": PATIENT_ID,
                "run_id": RUN_ID,
                "modality_index": idx,
                "modality_name": rec["modality_name"],
                "path": rec["path"],
                "zip_member": rec["zip_member"],
                "source_kind": rec["source_kind"],
                "status": "loaded",
                "n_arrays": len(arrays),
                "array_shapes_first": ";".join(shapes),
                "file_size_bytes": rec["file_size_bytes"],
            })

    if len(images) == 0:
        images = [np.zeros((dynamic_image_h, dynamic_image_w), dtype=np.float32)]
        tokens = [np.zeros(dynamic_token_features, dtype=np.float32)]
        modality_names = ["no_loaded_dynamical_outputs"]
        report_rows.append({
            "patient_id": PATIENT_ID,
            "run_id": RUN_ID,
            "modality_index": 0,
            "modality_name": "no_loaded_dynamical_outputs",
            "path": patient_dir,
            "zip_member": "",
            "source_kind": "fallback",
            "status": "fallback_empty",
            "n_arrays": 0,
            "array_shapes_first": "",
            "file_size_bytes": 0,
        })

    image_bank = np.stack(images).astype(np.float32)
    token_bank = np.stack(tokens).astype(np.float32)
    mask = np.ones((len(images),), dtype=np.float32)
    source_report = pd.DataFrame(report_rows)

    log_msg(f"Finished processing in {time.time() - t0:.1f}s.")
    log_msg(f"Loaded modalities: {len(modality_names)}")
    log_msg(f"Source report rows: {len(source_report)}")

    return image_bank, token_bank, mask, modality_names, source_report

def add_or_update_patient_index(row):
    if os.path.exists(patient_index_csv):
        df = pd.read_csv(patient_index_csv)
    else:
        df = pd.DataFrame()
    if len(df) > 0 and "patient_id" in df.columns:
        same = (
            (df["patient_id"].astype(str) == str(row["patient_id"]))
            & (df["run_id"].astype(str) == str(row["run_id"]))
        )
        df = df[~same]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(patient_index_csv, index=False)
    return df

def encode_labels(labels):
    labels = [str(x) for x in labels]
    unique = sorted(set(labels))
    mapping = {lab: i for i, lab in enumerate(unique)}
    y = np.asarray([mapping[x] for x in labels], dtype=np.int64)
    return y, mapping

def load_patient_tensor(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def build_union_modality_registry(patient_bundles):
    names = []
    for b in patient_bundles:
        for name in b["modality_names"]:
            if name not in names:
                names.append(name)
    return names

def align_patient_to_registry(bundle, registry):
    m = len(registry)
    h = int(bundle["dynamical_image_bank"].shape[-2])
    w = int(bundle["dynamical_image_bank"].shape[-1])
    f = int(bundle["dynamical_tokens"].shape[-1])
    image_out = torch.zeros((m, h, w), dtype=torch.float32)
    token_out = torch.zeros((m, f), dtype=torch.float32)
    mask_out = torch.zeros((m,), dtype=torch.float32)
    local_name_to_idx = {name: i for i, name in enumerate(bundle["modality_names"])}
    for global_i, name in enumerate(registry):
        if name in local_name_to_idx:
            local_i = local_name_to_idx[name]
            image_out[global_i] = bundle["dynamical_image_bank"][local_i].float()
            token_out[global_i] = bundle["dynamical_tokens"][local_i].float()
            mask_out[global_i] = 1.0
    return image_out, token_out, mask_out

def run_add_patient():
    patient_safe = safe_filename(f"{PATIENT_ID}_{RUN_ID}")
    patient_tensor_path = os.path.join(patient_tensor_dir, f"{patient_safe}_all_dynamical_tensor.pt")
    source_report_path = os.path.join(source_report_dir, f"{patient_safe}_source_report.csv")

    if USE_EXISTING_PATIENT_TENSOR_IF_PRESENT and os.path.exists(patient_tensor_path):
        log_msg(f"Using existing patient tensor: {patient_tensor_path}")
        patient_bundle = load_patient_tensor(patient_tensor_path)
        modality_names = patient_bundle["modality_names"]
        source_report = pd.read_csv(source_report_path) if os.path.exists(source_report_path) else pd.DataFrame()
    else:
        image_bank, token_bank, mask, modality_names, source_report = build_patient_dynamical_tensors(
            PATIENT_DYNAMICAL_RESULTS_DIR
        )
        image_dtype = torch.float16 if save_float16_for_images else torch.float32
        patient_bundle = {
            "patient_id": PATIENT_ID,
            "label": LABEL,
            "site": SITE,
            "run_id": RUN_ID,
            "dynamical_results_dir": PATIENT_DYNAMICAL_RESULTS_DIR,
            "speed_mode": SPEED_MODE,
            "dynamical_image_bank": torch.tensor(image_bank, dtype=image_dtype),
            "dynamical_tokens": torch.tensor(token_bank, dtype=torch.float32),
            "dynamical_mask": torch.tensor(mask, dtype=torch.float32),
            "modality_names": modality_names,
            "dynamic_image_h": dynamic_image_h,
            "dynamic_image_w": dynamic_image_w,
            "dynamic_token_features": dynamic_token_features,
        }
        torch.save(patient_bundle, patient_tensor_path)
        source_report.to_csv(source_report_path, index=False)

    index_row = {
        "patient_id": PATIENT_ID,
        "label": LABEL,
        "site": SITE,
        "run_id": RUN_ID,
        "dynamical_results_dir": PATIENT_DYNAMICAL_RESULTS_DIR,
        "patient_tensor_path": patient_tensor_path,
        "source_report_path": source_report_path,
        "speed_mode": SPEED_MODE,
        "n_modalities_loaded": int(len(modality_names)),
        "dynamic_image_h": int(dynamic_image_h),
        "dynamic_image_w": int(dynamic_image_w),
        "dynamic_token_features": int(dynamic_token_features),
    }
    index_df = add_or_update_patient_index(index_row)

    loaded = source_report[source_report["status"] == "loaded"] if not source_report.empty and "status" in source_report.columns else pd.DataFrame()
    skipped = source_report[source_report["status"] != "loaded"] if not source_report.empty and "status" in source_report.columns else pd.DataFrame()

    log_msg("\n" + "=" * 100)
    log_msg("PATIENT ALL-DYNAMICAL TENSOR CREATED / UPDATED")
    log_msg("=" * 100)
    log_msg(f"Speed mode: {SPEED_MODE}")
    log_msg(f"Patient ID: {PATIENT_ID}")
    log_msg(f"Label: {LABEL}")
    log_msg(f"Site: {SITE}")
    log_msg(f"Run ID: {RUN_ID}")
    log_msg(f"Results dir: {PATIENT_DYNAMICAL_RESULTS_DIR}")
    log_msg(f"Patient tensor: {patient_tensor_path}")
    log_msg(f"Source report: {source_report_path}")
    log_msg(f"Patient index: {patient_index_csv}")
    log_msg(f"Modalities loaded: {len(modality_names)}")
    log_msg("\nTensor shapes:")
    log_msg(f"  dynamical_image_bank: {tuple(patient_bundle['dynamical_image_bank'].shape)} = modalities x H x W")
    log_msg(f"  dynamical_tokens: {tuple(patient_bundle['dynamical_tokens'].shape)} = modalities x token_features")
    log_msg(f"  dynamical_mask: {tuple(patient_bundle['dynamical_mask'].shape)}")
    log_msg(f"\nLoaded files/modalities: {len(loaded)}")
    log_msg(f"Skipped/unusable outputs: {len(skipped)}")

    if len(loaded) > 0:
        cols = ["modality_index", "modality_name", "source_kind", "path", "zip_member", "n_arrays"]
        existing_cols = [c for c in cols if c in loaded.columns]
        log_msg("\nFirst 30 loaded modalities:")
        log_msg(loaded[existing_cols].head(30).to_string(index=False))

    log_msg("\nCurrent patient index:")
    log_msg(index_df[["patient_id", "run_id", "label", "site", "speed_mode", "n_modalities_loaded"]].to_string(index=False))

def run_build_training_bundle():
    if not os.path.exists(patient_index_csv):
        raise FileNotFoundError(f"No patient index found: {patient_index_csv}. Run MODE='add_patient' for each patient first.")

    index_df = pd.read_csv(patient_index_csv)

    if index_df.empty:
        raise RuntimeError("Patient index is empty.")

    patient_bundles = []

    log_msg(f"Loading {len(index_df)} patient tensors...")

    for i, row in index_df.iterrows():
        p = row["patient_tensor_path"]
        log_msg(f"[{i + 1}/{len(index_df)}] {p}")
        if not os.path.exists(p):
            log_msg(f"Skipping missing tensor: {p}")
            continue
        patient_bundles.append(load_patient_tensor(p))

    if len(patient_bundles) == 0:
        raise RuntimeError("No patient tensors loaded.")

    log_msg("Building union modality registry...")
    registry = build_union_modality_registry(patient_bundles)
    log_msg(f"Union modalities: {len(registry)}")

    aligned_images = []
    aligned_tokens = []
    aligned_masks = []
    patient_ids = []
    labels = []
    sites = []
    run_ids = []

    for i, b in enumerate(patient_bundles):
        log_msg(f"Aligning patient {i + 1}/{len(patient_bundles)}: {b['patient_id']} / {b['run_id']}")
        img, tok, msk = align_patient_to_registry(b, registry)
        aligned_images.append(img)
        aligned_tokens.append(tok)
        aligned_masks.append(msk)
        patient_ids.append(str(b["patient_id"]))
        labels.append(str(b["label"]))
        sites.append(str(b["site"]))
        run_ids.append(str(b["run_id"]))

    dynamical_image_bank = torch.stack(aligned_images, dim=0)
    dynamical_tokens = torch.stack(aligned_tokens, dim=0)
    dynamical_mask = torch.stack(aligned_masks, dim=0)

    label_indices, label_to_index = encode_labels(labels)
    site_indices, site_to_index = encode_labels(sites)
    patient_indices, patient_to_index = encode_labels(patient_ids)

    bundle = {
        "dynamical_image_bank": dynamical_image_bank,
        "dynamical_tokens": dynamical_tokens,
        "dynamical_mask": dynamical_mask,
        "labels": torch.tensor(label_indices, dtype=torch.long),
        "sites": torch.tensor(site_indices, dtype=torch.long),
        "subjects": torch.tensor(patient_indices, dtype=torch.long),
        "patient_ids_str": patient_ids,
        "labels_str": labels,
        "sites_str": sites,
        "run_ids_str": run_ids,
    }

    sample_manifest = pd.DataFrame({
        "sample_index": np.arange(len(patient_ids)),
        "patient_id": patient_ids,
        "subject_id": patient_ids,
        "run_id": run_ids,
        "label": labels,
        "label_index": label_indices,
        "site": sites,
        "site_index": site_indices,
    })
    sample_manifest.to_csv(training_manifest_path, index=False)

    metadata = {
        "speed_mode_last_run": SPEED_MODE,
        "n_patients_or_runs": int(len(patient_ids)),
        "n_modalities_union": int(len(registry)),
        "dynamic_image_h": int(dynamic_image_h),
        "dynamic_image_w": int(dynamic_image_w),
        "dynamic_token_features": int(dynamic_token_features),
        "tensor_shapes": {
            "dynamical_image_bank": list(dynamical_image_bank.shape),
            "dynamical_tokens": list(dynamical_tokens.shape),
            "dynamical_mask": list(dynamical_mask.shape),
            "labels": list(bundle["labels"].shape),
            "sites": list(bundle["sites"].shape),
            "subjects": list(bundle["subjects"].shape),
        },
        "label_to_index": label_to_index,
        "site_to_index": site_to_index,
        "patient_to_index": patient_to_index,
        "modality_registry": registry,
        "patient_index_csv": patient_index_csv,
        "sample_manifest_path": training_manifest_path,
        "note": (
            "Each sample is one patient/run-level dynamical-systems results folder. "
            "Ultra/fast modes intentionally skip or limit slow output types. "
            "Use full mode only when you need the most exhaustive scan."
        ),
    }

    log_msg("Saving training bundle...")
    torch.save(bundle, training_bundle_path)

    with open(training_metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    with open(modality_registry_path, "w") as f:
        json.dump(registry, f, indent=2)

    log_msg("\n" + "=" * 100)
    log_msg("PATIENT-LEVEL ALL-DYNAMICAL TRAINING BUNDLE CREATED")
    log_msg("=" * 100)
    log_msg(f"Training bundle: {training_bundle_path}")
    log_msg(f"Metadata: {training_metadata_path}")
    log_msg(f"Sample manifest: {training_manifest_path}")
    log_msg(f"Modality registry: {modality_registry_path}")
    log_msg("\nTensor shapes:")
    for k, v in bundle.items():
        if isinstance(v, torch.Tensor):
            log_msg(f"  {k}: {tuple(v.shape)}")
    log_msg("\nLabels:")
    for k, v in label_to_index.items():
        log_msg(f"  {k}: {v}")
    log_msg("\nSamples:")
    log_msg(sample_manifest.to_string(index=False))

log_msg("=" * 100)
log_msg("PATIENT-LEVEL DYNAMICAL TENSOR CREATION")
log_msg("=" * 100)
log_msg(f"MODE: {MODE}")
log_msg(f"SPEED_MODE: {SPEED_MODE}")
log_msg(f"Image size: {dynamic_image_h} x {dynamic_image_w}")
log_msg(f"Token features: {dynamic_token_features}")
log_msg(f"Include images: {include_images}")
log_msg(f"Include text/html/log/md: {include_text_like}")
log_msg(f"Include ZIP contents: {include_zip}")
log_msg(f"Max files to scan: {max_files_to_scan}")
log_msg(f"Max arrays per file: {max_arrays_per_file}")
log_msg(f"Max values per array: {max_values_per_array}")
log_msg("=" * 100)

if MODE == "add_patient":
    run_add_patient()
elif MODE == "build_training_bundle":
    run_build_training_bundle()
else:
    raise ValueError("MODE must be either 'add_patient' or 'build_training_bundle'.")

PATIENT-LEVEL DYNAMICAL TENSOR CREATION
MODE: add_patient
SPEED_MODE: ultra
Image size: 32 x 32
Token features: 32
Include images: False
Include text/html/log/md: False
Include ZIP contents: False
Max files to scan: 350
Max arrays per file: 8
Max values per array: 120000
Scanning folder: /home/a/projects/Complete-Neural-Signal-Analysis/results
SPEED_MODE: ultra
Allowed extensions: ['.csv', '.json', '.npy', '.npz']
Limiting scan from 545 files to 350 smallest files for ultra mode.
Candidate files: 350
Skipped by size: 6
Skipped excluded/generated folders: 9
Beginning file processing...
[1/350] loaded=0, skipped=0, elapsed=0.0s | lyap_spectrum_20260313_133207.csv
[10/350] loaded=6, skipped=3, elapsed=0.0s | custom_graph_communities.csv
[20/350] loaded=16, skipped=3, elapsed=0.1s | lyapunov_stability_by_dimension_summary.csv
[30/350] loaded=26, skipped=3, elapsed=0.2s | rosenstein_moreS_qc_C3_m10_tau8_nS199_20260603_154926.csv
[40/350] loaded=36, skipped=3, elapsed=0.3s | near_flat_router

In [8]:
import os
import json
import torch
import numpy as np
import pandas as pd

base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

dataset_bank_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_tensor_bank")
patient_index_csv = os.path.join(dataset_bank_dir, "patient_tensor_index.csv")

training_bundle_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_bundle.pt")
training_metadata_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_metadata.json")
training_manifest_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_sample_manifest.csv")
modality_registry_path = os.path.join(dataset_bank_dir, "modality_registry.json")

def load_patient_tensor(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def encode_labels(labels):
    labels = [str(x) for x in labels]
    unique = sorted(set(labels))
    mapping = {lab: i for i, lab in enumerate(unique)}
    y = np.asarray([mapping[x] for x in labels], dtype=np.int64)
    return y, mapping

def build_union_modality_registry(patient_bundles):
    names = []
    for b in patient_bundles:
        for name in b["modality_names"]:
            if name not in names:
                names.append(name)
    return names

def align_patient_to_registry(bundle, registry):
    m = len(registry)
    h = int(bundle["dynamical_image_bank"].shape[-2])
    w = int(bundle["dynamical_image_bank"].shape[-1])
    f = int(bundle["dynamical_tokens"].shape[-1])

    image_out = torch.zeros((m, h, w), dtype=torch.float32)
    token_out = torch.zeros((m, f), dtype=torch.float32)
    mask_out = torch.zeros((m,), dtype=torch.float32)

    local_name_to_idx = {name: i for i, name in enumerate(bundle["modality_names"])}

    for global_i, name in enumerate(registry):
        if name in local_name_to_idx:
            local_i = local_name_to_idx[name]
            image_out[global_i] = bundle["dynamical_image_bank"][local_i].float()
            token_out[global_i] = bundle["dynamical_tokens"][local_i].float()
            mask_out[global_i] = 1.0

    return image_out, token_out, mask_out

if not os.path.exists(patient_index_csv):
    raise FileNotFoundError(f"Missing patient index: {patient_index_csv}")

index_df = pd.read_csv(patient_index_csv)

if index_df.empty:
    raise RuntimeError("Patient index exists, but it is empty.")

patient_bundles = []

print("=" * 100)
print("BUILDING PATIENT-LEVEL ALL-DYNAMICAL TRAINING BUNDLE")
print("=" * 100)
print(f"Patient index: {patient_index_csv}")
print(f"Patients/runs listed: {len(index_df)}")

for i, row in index_df.iterrows():
    tensor_path = row["patient_tensor_path"]

    print(f"[{i + 1}/{len(index_df)}] Loading: {tensor_path}")

    if not os.path.exists(tensor_path):
        print(f"  Missing, skipped: {tensor_path}")
        continue

    b = load_patient_tensor(tensor_path)
    patient_bundles.append(b)

if len(patient_bundles) == 0:
    raise RuntimeError("No patient tensors were loaded. Check patient_tensor_path values in patient_tensor_index.csv.")

print("\nBuilding union modality registry...")
registry = build_union_modality_registry(patient_bundles)
print(f"Union modalities: {len(registry)}")

aligned_images = []
aligned_tokens = []
aligned_masks = []
patient_ids = []
labels = []
sites = []
run_ids = []
source_dirs = []

for i, b in enumerate(patient_bundles):
    print(f"[{i + 1}/{len(patient_bundles)}] Aligning: {b['patient_id']} / {b['run_id']}")

    img, tok, msk = align_patient_to_registry(b, registry)

    aligned_images.append(img)
    aligned_tokens.append(tok)
    aligned_masks.append(msk)

    patient_ids.append(str(b["patient_id"]))
    labels.append(str(b["label"]))
    sites.append(str(b["site"]))
    run_ids.append(str(b["run_id"]))
    source_dirs.append(str(b.get("dynamical_results_dir", "")))

dynamical_image_bank = torch.stack(aligned_images, dim=0)
dynamical_tokens = torch.stack(aligned_tokens, dim=0)
dynamical_mask = torch.stack(aligned_masks, dim=0)

label_indices, label_to_index = encode_labels(labels)
site_indices, site_to_index = encode_labels(sites)
patient_indices, patient_to_index = encode_labels(patient_ids)

bundle = {
    "dynamical_image_bank": dynamical_image_bank,
    "dynamical_tokens": dynamical_tokens,
    "dynamical_mask": dynamical_mask,
    "labels": torch.tensor(label_indices, dtype=torch.long),
    "sites": torch.tensor(site_indices, dtype=torch.long),
    "subjects": torch.tensor(patient_indices, dtype=torch.long),
    "patient_ids_str": patient_ids,
    "labels_str": labels,
    "sites_str": sites,
    "run_ids_str": run_ids,
}

sample_manifest = pd.DataFrame({
    "sample_index": np.arange(len(patient_ids)),
    "patient_id": patient_ids,
    "subject_id": patient_ids,
    "run_id": run_ids,
    "label": labels,
    "label_index": label_indices,
    "site": sites,
    "site_index": site_indices,
    "dynamical_results_dir": source_dirs,
})

sample_manifest.to_csv(training_manifest_path, index=False)

metadata = {
    "n_patients_or_runs": int(len(patient_ids)),
    "n_modalities_union": int(len(registry)),
    "dynamic_image_h": int(dynamical_image_bank.shape[-2]),
    "dynamic_image_w": int(dynamical_image_bank.shape[-1]),
    "dynamic_token_features": int(dynamical_tokens.shape[-1]),
    "tensor_shapes": {
        "dynamical_image_bank": list(dynamical_image_bank.shape),
        "dynamical_tokens": list(dynamical_tokens.shape),
        "dynamical_mask": list(dynamical_mask.shape),
        "labels": list(bundle["labels"].shape),
        "sites": list(bundle["sites"].shape),
        "subjects": list(bundle["subjects"].shape),
    },
    "label_to_index": label_to_index,
    "site_to_index": site_to_index,
    "patient_to_index": patient_to_index,
    "modality_registry": registry,
    "patient_index_csv": patient_index_csv,
    "sample_manifest_path": training_manifest_path,
    "note": "Each sample is one patient/run-level dynamical-systems result folder. Missing modalities are zero-filled and masked with dynamical_mask=0.",
}

torch.save(bundle, training_bundle_path)

with open(training_metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

with open(modality_registry_path, "w") as f:
    json.dump(registry, f, indent=2)

print("\n" + "=" * 100)
print("TRAINING BUNDLE CREATED")
print("=" * 100)
print(f"Training bundle: {training_bundle_path}")
print(f"Metadata: {training_metadata_path}")
print(f"Sample manifest: {training_manifest_path}")
print(f"Modality registry: {modality_registry_path}")

print("\nTensor shapes:")
for k, v in bundle.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}")

print("\nLabels:")
for k, v in label_to_index.items():
    print(f"  {k}: {v}")

print("\nSamples:")
print(sample_manifest.to_string(index=False))

print("\nNext step:")
if len(set(labels)) < 2:
    print("You only have one class so far. The neural-net block can run, but it will NOT truly train disease detection yet.")
    print("Add at least one disease patient/run and one control patient/run for real binary training.")
else:
    print("You have more than one class. Run the neural-net training block next.")

BUILDING PATIENT-LEVEL ALL-DYNAMICAL TRAINING BUNDLE
Patient index: /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_tensor_index.csv
Patients/runs listed: 1
[1/1] Loading: /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_tensors/sub001_run1_all_dynamical_tensor.pt

Building union modality registry...
Union modalities: 347
[1/1] Aligning: sub001 / run1

TRAINING BUNDLE CREATED
Training bundle: /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_level_all_dynamical_training_bundle.pt
Metadata: /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_level_all_dynamical_training_metadata.json
Sample manifest: /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_level_all_dynamical_sample_manifest.csv
Modality registry: /home/a/

### Neural Net

In [9]:
import os
import pandas as pd

base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

dataset_bank_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_tensor_bank")
model_out_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_neural_net")

paths = {
    "patient index": os.path.join(dataset_bank_dir, "patient_tensor_index.csv"),
    "training bundle": os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_bundle.pt"),
    "training metadata": os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_metadata.json"),
    "sample manifest": os.path.join(dataset_bank_dir, "patient_level_all_dynamical_sample_manifest.csv"),
    "predictions CSV": os.path.join(model_out_dir, "patient_level_predictions.csv"),
    "subject summary CSV": os.path.join(model_out_dir, "patient_level_subject_summary.csv"),
    "run summary JSON": os.path.join(model_out_dir, "run_summary.json"),
    "training history CSV": os.path.join(model_out_dir, "training_history.csv"),
    "embeddings PT": os.path.join(model_out_dir, "patient_level_embeddings.pt"),
    "logits PT": os.path.join(model_out_dir, "patient_level_logits.pt"),
}

print("=" * 90)
print("PIPELINE FILE CHECK")
print("=" * 90)

for name, path in paths.items():
    print(f"{name:22s}: {'FOUND' if os.path.exists(path) else 'MISSING'} | {path}")

print("\n" + "=" * 90)
print("NEXT STEP")
print("=" * 90)

if not os.path.exists(paths["patient index"]):
    print("You have not added any patient yet.")
    print("Run tensor block with MODE = 'add_patient' for each patient/run.")

elif not os.path.exists(paths["training bundle"]):
    print("Patient tensors exist, but the combined training bundle does not.")
    print("Run tensor block once with MODE = 'build_training_bundle'.")

elif not os.path.exists(paths["predictions CSV"]):
    print("Training bundle exists, but neural-net outputs do not.")
    print("Run the neural-net training block next.")
    print("After that, run the expert plotting block.")

else:
    print("Neural-net outputs exist.")
    print("You can run the expert plotting block now.")

if os.path.exists(paths["patient index"]):
    print("\nPatient index preview:")
    print(pd.read_csv(paths["patient index"]).to_string(index=False))

if os.path.exists(paths["sample manifest"]):
    print("\nTraining sample manifest preview:")
    print(pd.read_csv(paths["sample manifest"]).to_string(index=False))

PIPELINE FILE CHECK
patient index         : FOUND | /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_tensor_index.csv
training bundle       : FOUND | /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_level_all_dynamical_training_bundle.pt
training metadata     : FOUND | /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_level_all_dynamical_training_metadata.json
sample manifest       : FOUND | /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_tensor_bank/patient_level_all_dynamical_sample_manifest.csv
predictions CSV       : MISSING | /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_neural_net/patient_level_predictions.csv
subject summary CSV   : MISSING | /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_neural_net/pat

In [10]:
import os
import json
import math
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

warnings.filterwarnings("ignore")

base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

dataset_bank_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_tensor_bank")
training_bundle_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_bundle.pt")
training_metadata_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_metadata.json")
training_manifest_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_sample_manifest.csv")

out_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_neural_net")
os.makedirs(out_dir, exist_ok=True)

SEED = 42
BATCH_SIZE = 4
EPOCHS = 100
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 128
N_HEADS = 4
DROPOUT = 0.20
FUSION_LAYERS = 3
MODALITY_LAYERS = 2
MAX_MODALITIES_FOR_MODEL = 256
VAL_FRACTION = 0.20
SITE_ADVERSARIAL_WEIGHT = 0.03
GRAD_CLIP = 3.0
SAVE_EVERYTHING = True

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not os.path.exists(training_bundle_path):
    raise FileNotFoundError(
        f"Missing training bundle:\n{training_bundle_path}\n\n"
        "Run the tensor block with MODE='build_training_bundle' first."
    )

try:
    bundle = torch.load(training_bundle_path, map_location="cpu", weights_only=False)
except TypeError:
    bundle = torch.load(training_bundle_path, map_location="cpu")

metadata = {}
if os.path.exists(training_metadata_path):
    with open(training_metadata_path, "r") as f:
        metadata = json.load(f)

sample_manifest = pd.read_csv(training_manifest_path) if os.path.exists(training_manifest_path) else None

required_keys = [
    "dynamical_image_bank",
    "dynamical_tokens",
    "dynamical_mask",
    "labels",
    "sites",
    "subjects",
]

missing = [k for k in required_keys if k not in bundle]
if missing:
    raise KeyError(f"Training bundle is missing required keys: {missing}")

n_samples = int(bundle["labels"].shape[0])
n_modalities_original = int(bundle["dynamical_image_bank"].shape[1])

if n_modalities_original > MAX_MODALITIES_FOR_MODEL:
    keep_idx = torch.linspace(
        0,
        n_modalities_original - 1,
        MAX_MODALITIES_FOR_MODEL,
    ).long()
    bundle["dynamical_image_bank"] = bundle["dynamical_image_bank"][:, keep_idx]
    bundle["dynamical_tokens"] = bundle["dynamical_tokens"][:, keep_idx]
    bundle["dynamical_mask"] = bundle["dynamical_mask"][:, keep_idx]
else:
    keep_idx = torch.arange(n_modalities_original).long()

labels = bundle["labels"].long()
sites = bundle["sites"].long()
subjects = bundle["subjects"].long()

actual_num_classes = int(labels.max().item()) + 1 if labels.numel() else 1
actual_num_sites = int(sites.max().item()) + 1 if sites.numel() else 1

num_classes = actual_num_classes if actual_num_classes > 1 else 2
num_sites = actual_num_sites if actual_num_sites > 1 else 2

train_enabled = actual_num_classes > 1 and n_samples >= 4

label_to_index = metadata.get("label_to_index", {})
index_to_label = {int(v): str(k) for k, v in label_to_index.items()} if label_to_index else {i: str(i) for i in range(num_classes)}

positive_keywords = [
    "disease",
    "pd",
    "parkinson",
    "ad",
    "alz",
    "mci",
    "epilepsy",
    "patient",
    "positive",
    "case",
]

positive_class_index = None
for idx, lab in index_to_label.items():
    if any(k in str(lab).lower() for k in positive_keywords):
        positive_class_index = int(idx)
        break

if positive_class_index is None:
    positive_class_index = 1 if num_classes > 1 else 0

if positive_class_index >= num_classes:
    positive_class_index = num_classes - 1

def get_bundle_string_list(key, fallback_prefix):
    if key in bundle and isinstance(bundle[key], list):
        return [str(x) for x in bundle[key]]
    return [f"{fallback_prefix}_{i:04d}" for i in range(n_samples)]

patient_ids_str = get_bundle_string_list("patient_ids_str", "patient")
labels_str = get_bundle_string_list("labels_str", "label")
sites_str = get_bundle_string_list("sites_str", "site")
run_ids_str = get_bundle_string_list("run_ids_str", "run")

print("=" * 100)
print("LOADED PATIENT-LEVEL ALL-DYNAMICAL TRAINING BUNDLE")
print("=" * 100)
print(f"Device: {device}")
print(f"Samples / patient-runs: {n_samples}")
print(f"Original modalities: {n_modalities_original}")
print(f"Model modalities: {bundle['dynamical_image_bank'].shape[1]}")
print(f"Actual classes in data: {actual_num_classes}")
print(f"Model output classes: {num_classes}")
print(f"Actual sites in data: {actual_num_sites}")
print(f"Model site classes: {num_sites}")
print(f"Training enabled: {train_enabled}")
print(f"Positive / disease class index: {positive_class_index}")
print(f"Positive / disease class label: {index_to_label.get(positive_class_index, positive_class_index)}")
print("\nTensor shapes:")
for k in required_keys:
    print(f"  {k}: {tuple(bundle[k].shape)}")

class PatientDynamicalDataset(Dataset):
    def __init__(self, bundle):
        self.bundle = bundle
        self.n = int(bundle["labels"].shape[0])

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return {
            "dynamical_image_bank": self.bundle["dynamical_image_bank"][idx],
            "dynamical_tokens": self.bundle["dynamical_tokens"][idx],
            "dynamical_mask": self.bundle["dynamical_mask"][idx],
            "labels": self.bundle["labels"][idx],
            "sites": self.bundle["sites"][idx],
            "subjects": self.bundle["subjects"][idx],
            "sample_index": torch.tensor(idx, dtype=torch.long),
        }

def make_train_val_indices(labels, val_fraction=0.2):
    n = len(labels)
    indices = np.arange(n)
    y = labels.cpu().numpy().astype(int)
    rng = np.random.default_rng(SEED)

    if n < 4 or len(np.unique(y)) < 2:
        return indices, indices

    train_idx = []
    val_idx = []

    for c in sorted(np.unique(y)):
        class_idx = indices[y == c]
        rng.shuffle(class_idx)

        if len(class_idx) >= 2:
            n_val_c = max(1, int(round(len(class_idx) * val_fraction)))
        else:
            n_val_c = 0

        val_idx.extend(class_idx[:n_val_c].tolist())
        train_idx.extend(class_idx[n_val_c:].tolist())

    if len(train_idx) == 0 or len(val_idx) == 0:
        shuffled = indices.copy()
        rng.shuffle(shuffled)
        n_val = max(1, int(round(n * val_fraction)))
        val_idx = shuffled[:n_val].tolist()
        train_idx = shuffled[n_val:].tolist()

    if len(train_idx) == 0:
        train_idx = val_idx.copy()

    return np.asarray(train_idx, dtype=int), np.asarray(val_idx, dtype=int)

class GradientReverseFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

def grad_reverse(x, lambda_=1.0):
    return GradientReverseFunction.apply(x, lambda_)

class SharedModalityImageEncoder(nn.Module):
    def __init__(self, d_model, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 24, kernel_size=3, padding=1),
            nn.BatchNorm2d(24),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),

            nn.Conv2d(24, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),

            nn.Conv2d(48, 80, kernel_size=3, padding=1),
            nn.BatchNorm2d(80),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(80 * 2 * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.proj(self.cnn(x))

class ModalityFusionEncoder(nn.Module):
    def __init__(self, n_modalities, token_features, d_model, n_layers=2, dropout=0.2):
        super().__init__()
        self.image_encoder = SharedModalityImageEncoder(d_model=d_model, dropout=dropout)

        self.token_proj = nn.Sequential(
            nn.Linear(token_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.modality_pos = nn.Parameter(torch.randn(1, n_modalities, d_model) * 0.02)

        self.gate = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid(),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, images, tokens, mask):
        images = images.float()
        tokens = tokens.float()
        mask = mask.float()

        b, m, h, w = images.shape

        img_flat = images.reshape(b * m, 1, h, w)
        img_tok = self.image_encoder(img_flat).reshape(b, m, -1)

        stat_tok = self.token_proj(tokens)

        gate = self.gate(torch.cat([img_tok, stat_tok], dim=-1))
        fused = gate * img_tok + (1.0 - gate) * stat_tok

        fused = fused + self.modality_pos[:, :m]

        key_padding_mask = mask <= 0
        encoded = self.transformer(fused, src_key_padding_mask=key_padding_mask)

        valid = mask.unsqueeze(-1)
        pooled = (encoded * valid).sum(dim=1) / (valid.sum(dim=1) + 1e-6)

        return self.norm(pooled), self.norm(encoded)

class CrossAttentionDiseaseHead(nn.Module):
    def __init__(self, d_model, n_layers=3, dropout=0.2):
        super().__init__()
        self.disease_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, modality_tokens, mask):
        b = modality_tokens.shape[0]

        cls = self.disease_token.expand(b, -1, -1)
        tokens = torch.cat([cls, modality_tokens], dim=1)

        cls_mask = torch.zeros((b, 1), device=mask.device, dtype=torch.bool)
        key_padding_mask = torch.cat([cls_mask, mask <= 0], dim=1)

        out = self.transformer(tokens, src_key_padding_mask=key_padding_mask)

        return self.norm(out[:, 0]), self.norm(out[:, 1:])

class PatientLevelNeuroDynaNet(nn.Module):
    def __init__(self, n_modalities, token_features, num_classes, num_sites, d_model=128, dropout=0.2):
        super().__init__()

        self.modality_encoder = ModalityFusionEncoder(
            n_modalities=n_modalities,
            token_features=token_features,
            d_model=d_model,
            n_layers=MODALITY_LAYERS,
            dropout=dropout,
        )

        self.fusion_head = CrossAttentionDiseaseHead(
            d_model=d_model,
            n_layers=FUSION_LAYERS,
            dropout=dropout,
        )

        self.disease_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        self.site_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_sites),
        )

        self.uncertainty_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, batch, grl_lambda=0.0):
        pooled, modality_context = self.modality_encoder(
            batch["dynamical_image_bank"],
            batch["dynamical_tokens"],
            batch["dynamical_mask"],
        )

        disease_repr, context = self.fusion_head(
            modality_context,
            batch["dynamical_mask"],
        )

        disease_repr = disease_repr + pooled

        disease_logits = self.disease_head(disease_repr)
        site_logits = self.site_head(grad_reverse(disease_repr, grl_lambda))
        uncertainty = self.uncertainty_head(disease_repr).squeeze(-1)

        return {
            "disease_logits": disease_logits,
            "site_logits": site_logits,
            "uncertainty": uncertainty,
            "embedding": disease_repr,
            "modality_context": context,
        }

def move_batch(batch, device):
    out = {}

    for k, v in batch.items():
        if k in ["labels", "sites", "subjects", "sample_index"]:
            out[k] = v.to(device=device, dtype=torch.long)
        else:
            out[k] = v.to(device=device)

    return out

def class_weights(y, num_classes):
    y = y.cpu().numpy().astype(int)
    counts = np.bincount(y, minlength=num_classes).astype(float)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

def accuracy(logits, y):
    return (torch.argmax(logits, dim=1) == y).float().mean().item()

dataset = PatientDynamicalDataset(bundle)
train_idx, val_idx = make_train_val_indices(labels, VAL_FRACTION)

train_loader = DataLoader(
    Subset(dataset, train_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    Subset(dataset, val_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

full_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

model = PatientLevelNeuroDynaNet(
    n_modalities=int(bundle["dynamical_image_bank"].shape[1]),
    token_features=int(bundle["dynamical_tokens"].shape[-1]),
    num_classes=num_classes,
    num_sites=num_sites,
    d_model=D_MODEL,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 100)
print("PATIENT-LEVEL NEURODYNAMICAL NEURAL NET CREATED")
print("=" * 100)
print(f"Parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")
print(f"Train samples: {len(train_idx)}")
print(f"Validation samples: {len(val_idx)}")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(EPOCHS, 1),
)

disease_loss_fn = nn.CrossEntropyLoss(
    weight=class_weights(labels[train_idx], num_classes).to(device),
    label_smoothing=0.03,
)

site_loss_fn = nn.CrossEntropyLoss(
    weight=class_weights(sites[train_idx], num_sites).to(device),
)

history = []
best_val_loss = float("inf")

best_model_path = os.path.join(out_dir, "best_patient_level_neurodyna_model.pt")
last_model_path = os.path.join(out_dir, "last_patient_level_neurodyna_model.pt")

def evaluate(loader):
    model.eval()

    losses = []
    accs = []
    site_accs = []

    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch, device)
            out = model(batch, grl_lambda=0.0)

            if actual_num_classes > 1:
                loss = disease_loss_fn(out["disease_logits"], batch["labels"])
                losses.append(float(loss.item()))
                accs.append(accuracy(out["disease_logits"], batch["labels"]))

            if actual_num_sites > 1:
                site_accs.append(accuracy(out["site_logits"], batch["sites"]))

    return {
        "loss": float(np.mean(losses)) if losses else np.nan,
        "accuracy": float(np.mean(accs)) if accs else np.nan,
        "site_accuracy": float(np.mean(site_accs)) if site_accs else np.nan,
    }

if train_enabled:
    print("\nTraining started.")
    total_steps = max(1, EPOCHS * len(train_loader))
    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()

        train_losses = []
        train_accs = []

        for batch in train_loader:
            batch = move_batch(batch, device)

            p = global_step / total_steps
            grl_lambda = float(2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)

            optimizer.zero_grad(set_to_none=True)

            out = model(batch, grl_lambda=grl_lambda)

            disease_loss = disease_loss_fn(out["disease_logits"], batch["labels"])
            loss = disease_loss

            if actual_num_sites > 1:
                site_loss = site_loss_fn(out["site_logits"], batch["sites"])
                loss = loss + SITE_ADVERSARIAL_WEIGHT * site_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            train_losses.append(float(loss.item()))
            train_accs.append(accuracy(out["disease_logits"], batch["labels"]))

            global_step += 1

        scheduler.step()

        val_metrics = evaluate(val_loader)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)) if train_losses else np.nan,
            "train_accuracy": float(np.mean(train_accs)) if train_accs else np.nan,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_site_accuracy": val_metrics["site_accuracy"],
            "lr": float(optimizer.param_groups[0]["lr"]),
            "elapsed_sec": time.time() - t0,
        }

        history.append(row)

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"train_loss={row['train_loss']:.4f}, train_acc={row['train_accuracy']:.3f} | "
            f"val_loss={row['val_loss']:.4f}, val_acc={row['val_accuracy']:.3f} | "
            f"lr={row['lr']:.2e} | {row['elapsed_sec']:.1f}s"
        )

        if np.isfinite(row["val_loss"]) and row["val_loss"] < best_val_loss:
            best_val_loss = row["val_loss"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "metadata": metadata,
                    "config": {
                        "D_MODEL": D_MODEL,
                        "N_HEADS": N_HEADS,
                        "DROPOUT": DROPOUT,
                        "num_classes": num_classes,
                        "num_sites": num_sites,
                        "positive_class_index": positive_class_index,
                        "MAX_MODALITIES_FOR_MODEL": MAX_MODALITIES_FOR_MODEL,
                    },
                    "kept_modality_indices": keep_idx.tolist(),
                },
                best_model_path,
            )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "metadata": metadata,
            "config": {
                "D_MODEL": D_MODEL,
                "N_HEADS": N_HEADS,
                "DROPOUT": DROPOUT,
                "num_classes": num_classes,
                "num_sites": num_sites,
                "positive_class_index": positive_class_index,
                "MAX_MODALITIES_FOR_MODEL": MAX_MODALITIES_FOR_MODEL,
            },
            "kept_modality_indices": keep_idx.tolist(),
        },
        last_model_path,
    )

else:
    print("\nTraining skipped.")
    print("Reason: need at least 2 label classes and at least 4 patient/run samples.")
    print("The model will still run inference and save predictions, logits, and embeddings.")

def collect_outputs(loader):
    model.eval()

    rows = []
    embeddings = []
    logits_all = []
    site_logits_all = []
    uncertainty_all = []

    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch, device)
            out = model(batch, grl_lambda=0.0)

            logits = out["disease_logits"]
            probs = torch.softmax(logits, dim=1)

            if positive_class_index < probs.shape[1]:
                disease_prob = probs[:, positive_class_index]
            else:
                disease_prob = probs[:, -1]

            pred = torch.argmax(probs, dim=1)

            for i in range(logits.shape[0]):
                sample_index = int(batch["sample_index"][i].cpu().item())

                row = {
                    "sample_index": sample_index,
                    "patient_id": patient_ids_str[sample_index] if sample_index < len(patient_ids_str) else f"patient_{sample_index:04d}",
                    "subject_id": patient_ids_str[sample_index] if sample_index < len(patient_ids_str) else f"patient_{sample_index:04d}",
                    "run_id": run_ids_str[sample_index] if sample_index < len(run_ids_str) else "run",
                    "label": labels_str[sample_index] if sample_index < len(labels_str) else str(int(batch["labels"][i].cpu().item())),
                    "site": sites_str[sample_index] if sample_index < len(sites_str) else str(int(batch["sites"][i].cpu().item())),
                    "label_index": int(batch["labels"][i].cpu().item()),
                    "site_index": int(batch["sites"][i].cpu().item()),
                    "subject_index": int(batch["subjects"][i].cpu().item()),
                    "predicted_class_index": int(pred[i].cpu().item()),
                    "predicted_class_label": index_to_label.get(int(pred[i].cpu().item()), str(int(pred[i].cpu().item()))),
                    "disease_probability": float(disease_prob[i].cpu().item()),
                    "uncertainty_score": float(out["uncertainty"][i].cpu().item()),
                }

                for c in range(probs.shape[1]):
                    row[f"class_probability_{c}"] = float(probs[i, c].cpu().item())

                rows.append(row)

            embeddings.append(out["embedding"].detach().cpu())
            logits_all.append(logits.detach().cpu())
            site_logits_all.append(out["site_logits"].detach().cpu())
            uncertainty_all.append(out["uncertainty"].detach().cpu())

    return (
        pd.DataFrame(rows).sort_values("sample_index"),
        torch.cat(embeddings, dim=0),
        torch.cat(logits_all, dim=0),
        torch.cat(site_logits_all, dim=0),
        torch.cat(uncertainty_all, dim=0),
    )

pred_df, embeddings, logits, site_logits, uncertainty_scores = collect_outputs(full_loader)

subject_summary = (
    pred_df.groupby(["patient_id", "subject_id", "label", "site"], dropna=False)
    .agg(
        n_runs=("sample_index", "count"),
        disease_probability_mean=("disease_probability", "mean"),
        disease_probability_median=("disease_probability", "median"),
        disease_probability_max=("disease_probability", "max"),
        disease_probability_min=("disease_probability", "min"),
        uncertainty_mean=("uncertainty_score", "mean"),
        uncertainty_median=("uncertainty_score", "median"),
    )
    .reset_index()
)

pred_csv_path = os.path.join(out_dir, "patient_level_predictions.csv")
subject_csv_path = os.path.join(out_dir, "patient_level_subject_summary.csv")
embeddings_path = os.path.join(out_dir, "patient_level_embeddings.pt")
logits_path = os.path.join(out_dir, "patient_level_logits.pt")
site_logits_path = os.path.join(out_dir, "patient_level_site_logits.pt")
uncertainty_path = os.path.join(out_dir, "patient_level_uncertainty.pt")
history_csv_path = os.path.join(out_dir, "training_history.csv")
run_summary_path = os.path.join(out_dir, "run_summary.json")

pred_df.to_csv(pred_csv_path, index=False)
subject_summary.to_csv(subject_csv_path, index=False)
pd.DataFrame(history).to_csv(history_csv_path, index=False)

torch.save(embeddings, embeddings_path)
torch.save(logits, logits_path)
torch.save(site_logits, site_logits_path)
torch.save(uncertainty_scores, uncertainty_path)

run_summary = {
    "device": str(device),
    "n_samples": int(n_samples),
    "n_params": int(n_params),
    "n_trainable_params": int(n_trainable),
    "training_enabled": bool(train_enabled),
    "actual_num_classes": int(actual_num_classes),
    "num_classes": int(num_classes),
    "actual_num_sites": int(actual_num_sites),
    "num_sites": int(num_sites),
    "positive_class_index": int(positive_class_index),
    "positive_class_label": index_to_label.get(positive_class_index, str(positive_class_index)),
    "original_modalities": int(n_modalities_original),
    "model_modalities": int(bundle["dynamical_image_bank"].shape[1]),
    "train_samples": int(len(train_idx)),
    "val_samples": int(len(val_idx)),
    "best_val_loss": float(best_val_loss) if np.isfinite(best_val_loss) else None,
    "best_model_path": best_model_path if os.path.exists(best_model_path) else None,
    "last_model_path": last_model_path if os.path.exists(last_model_path) else None,
    "predictions_csv": pred_csv_path,
    "subject_summary_csv": subject_csv_path,
    "embeddings_path": embeddings_path,
    "logits_path": logits_path,
    "site_logits_path": site_logits_path,
    "uncertainty_path": uncertainty_path,
    "history_csv": history_csv_path,
    "training_bundle_path": training_bundle_path,
    "training_metadata_path": training_metadata_path,
    "training_manifest_path": training_manifest_path,
}

with open(run_summary_path, "w") as f:
    json.dump(run_summary, f, indent=2)

print("\n" + "=" * 100)
print("PATIENT-LEVEL NEURODYNAMICAL MODEL RUN COMPLETE")
print("=" * 100)
print(f"Predictions CSV: {pred_csv_path}")
print(f"Subject summary CSV: {subject_csv_path}")
print(f"Embeddings: {embeddings_path}")
print(f"Logits: {logits_path}")
print(f"Site logits: {site_logits_path}")
print(f"Uncertainty: {uncertainty_path}")
print(f"History: {history_csv_path}")
print(f"Run summary: {run_summary_path}")

if os.path.exists(best_model_path):
    print(f"Best model: {best_model_path}")

if os.path.exists(last_model_path):
    print(f"Last model: {last_model_path}")

print("\nPatient-level disease probabilities:")
print(subject_summary.to_string(index=False))

print("\nImportant:")
if not train_enabled:
    print("The output files were created, but true disease training did not occur.")
    print("Add multiple control and disease patient/runs, rebuild the training bundle, then rerun this cell.")
else:
    print("Training completed. Now run the separate plotting cell.")

LOADED PATIENT-LEVEL ALL-DYNAMICAL TRAINING BUNDLE
Device: cuda
Samples / patient-runs: 1
Original modalities: 347
Model modalities: 256
Actual classes in data: 1
Model output classes: 2
Actual sites in data: 1
Model site classes: 2
Training enabled: False
Positive / disease class index: 1
Positive / disease class label: 1

Tensor shapes:
  dynamical_image_bank: (1, 256, 32, 32)
  dynamical_tokens: (1, 256, 32)
  dynamical_mask: (1, 256)
  labels: (1,)
  sites: (1,)
  subjects: (1,)

PATIENT-LEVEL NEURODYNAMICAL NEURAL NET CREATED
Parameters: 1,199,845
Trainable parameters: 1,199,845
Train samples: 1
Validation samples: 1

Training skipped.
Reason: need at least 2 label classes and at least 4 patient/run samples.
The model will still run inference and save predictions, logits, and embeddings.

PATIENT-LEVEL NEURODYNAMICAL MODEL RUN COMPLETE
Predictions CSV: /home/a/projects/Complete-Neural-Signal-Analysis/results/patient_level_all_dynamical_neural_net/patient_level_predictions.csv
Subj

### Plots

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    torch = None
    TORCH_AVAILABLE = False

base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

model_out_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_neural_net")
dataset_bank_dir = os.path.join(base_dir, "results", "patient_level_all_dynamical_tensor_bank")

pred_csv_path = os.path.join(model_out_dir, "patient_level_predictions.csv")
subject_csv_path = os.path.join(model_out_dir, "patient_level_subject_summary.csv")
history_csv_path = os.path.join(model_out_dir, "training_history.csv")
run_summary_path = os.path.join(model_out_dir, "run_summary.json")
embeddings_path = os.path.join(model_out_dir, "patient_level_embeddings.pt")
logits_path = os.path.join(model_out_dir, "patient_level_logits.pt")

training_bundle_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_bundle.pt")
training_metadata_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_training_metadata.json")
training_manifest_path = os.path.join(dataset_bank_dir, "patient_level_all_dynamical_sample_manifest.csv")
modality_registry_path = os.path.join(dataset_bank_dir, "modality_registry.json")
patient_index_csv = os.path.join(dataset_bank_dir, "patient_tensor_index.csv")

plot_dir = os.path.join(model_out_dir, "expert_neural_net_plots")
os.makedirs(plot_dir, exist_ok=True)

show_plots_in_notebook = True
save_plots_to_disk = True
dpi_save = 220

ACCENT = "#00FFFF"
ACCENT_SOFT = "#66FFFF"
ACCENT_DIM = "#008B8B"
WHITE = "#E8FFFF"
BLACK = "#000000"
WARN = "#FFD966"

CYAN_SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list(
    "cyan_sequential",
    [BLACK, "#003333", ACCENT_DIM, ACCENT, ACCENT_SOFT],
    N=256,
)

plt.rcParams.update({
    "figure.facecolor": BLACK,
    "axes.facecolor": BLACK,
    "savefig.facecolor": BLACK,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT_DIM,
    "grid.alpha": 0.25,
    "legend.facecolor": BLACK,
    "legend.edgecolor": ACCENT,
    "legend.labelcolor": ACCENT,
    "font.size": 10,
})

eps = 1e-12

def exists(path):
    return os.path.exists(path)

def load_json(path):
    if exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {}

def load_torch(path):
    if not TORCH_AVAILABLE or not exists(path):
        return None
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")
    except Exception as e:
        print(f"Could not load torch file: {path}")
        print(f"{type(e).__name__}: {e}")
        return None

def to_numpy(x):
    if x is None:
        return None
    if TORCH_AVAILABLE and isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def style_ax(ax, title=None, xlabel=None, ylabel=None, grid=True, heatmap=False):
    ax.set_facecolor(BLACK)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)
        spine.set_linewidth(1.05)
    ax.tick_params(axis="x", colors=ACCENT, labelcolor=ACCENT)
    ax.tick_params(axis="y", colors=ACCENT, labelcolor=ACCENT)
    if heatmap:
        ax.grid(False)
    elif grid:
        ax.grid(True, alpha=0.22, color=ACCENT_DIM, linewidth=0.65)
    else:
        ax.grid(False)
    if title is not None:
        ax.set_title(title, color=ACCENT, pad=12)
    if xlabel is not None:
        ax.set_xlabel(xlabel, color=ACCENT)
    if ylabel is not None:
        ax.set_ylabel(ylabel, color=ACCENT)

def style_cbar(cbar, label):
    cbar.set_label(label, color=ACCENT)
    cbar.outline.set_edgecolor(ACCENT)
    cbar.ax.tick_params(color=ACCENT)
    for lab in cbar.ax.get_yticklabels():
        lab.set_color(ACCENT)

def save_show(fig, filename):
    path = os.path.join(plot_dir, filename)
    fig.tight_layout()
    if save_plots_to_disk:
        fig.savefig(path, dpi=dpi_save, bbox_inches="tight", facecolor=BLACK)
    if show_plots_in_notebook:
        plt.show()
    plt.close(fig)
    return path

def finite_clean(x):
    x = np.asarray(x, dtype=float)
    finite = x[np.isfinite(x)]
    fill = np.nanmedian(finite) if finite.size else 0.0
    return np.nan_to_num(x, nan=fill, posinf=fill, neginf=fill)

def pca_2d(X):
    X = finite_clean(np.asarray(X, dtype=float))
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    X = X - np.mean(X, axis=0, keepdims=True)
    if X.shape[0] < 2:
        return np.zeros((X.shape[0], 2)), np.array([0.0, 0.0])
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    coords = U[:, :min(2, U.shape[1])] * S[:min(2, len(S))]
    if coords.shape[1] < 2:
        coords = np.column_stack([coords[:, 0], np.zeros(coords.shape[0])])
    var = S ** 2
    explained = var / (np.sum(var) + eps)
    if len(explained) < 2:
        explained = np.pad(explained, (0, 2 - len(explained)))
    return coords[:, :2], explained[:2]

def confusion_matrix_np(y_true, y_pred, classes):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    index = {c: i for i, c in enumerate(classes)}
    for t, p in zip(y_true, y_pred):
        if t in index and p in index:
            cm[index[t], index[p]] += 1
    return cm

def binary_curves(y_true_index, disease_prob, positive_index):
    y = (np.asarray(y_true_index, dtype=int) == int(positive_index)).astype(int)
    score = np.asarray(disease_prob, dtype=float)
    valid = np.isfinite(score)
    y = y[valid]
    score = score[valid]

    if len(np.unique(y)) < 2:
        return None

    order = np.argsort(-score)
    y_sorted = y[order]

    P = np.sum(y_sorted == 1)
    N = np.sum(y_sorted == 0)

    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)

    tpr = tp / (P + eps)
    fpr = fp / (N + eps)
    precision = tp / (tp + fp + eps)
    recall = tpr.copy()

    fpr = np.concatenate([[0.0], fpr, [1.0]])
    tpr = np.concatenate([[0.0], tpr, [1.0]])

    recall_pr = np.concatenate([[0.0], recall])
    precision_pr = np.concatenate([[precision[0] if len(precision) else 1.0], precision])

    roc_auc = float(np.trapz(tpr, fpr))
    order_pr = np.argsort(recall_pr)
    pr_auc = float(np.trapz(precision_pr[order_pr], recall_pr[order_pr]))

    thresholds = np.unique(score)
    rows = []

    for th in thresholds:
        pred = (score >= th).astype(int)

        TP = np.sum((pred == 1) & (y == 1))
        FP = np.sum((pred == 1) & (y == 0))
        TN = np.sum((pred == 0) & (y == 0))
        FN = np.sum((pred == 0) & (y == 1))

        sens = TP / (TP + FN + eps)
        spec = TN / (TN + FP + eps)
        prec = TP / (TP + FP + eps)
        f1 = 2 * prec * sens / (prec + sens + eps)
        bal = 0.5 * (sens + spec)

        rows.append([th, sens, spec, prec, f1, bal])

    threshold_df = pd.DataFrame(
        rows,
        columns=["threshold", "sensitivity", "specificity", "precision", "f1", "balanced_accuracy"],
    )

    return {
        "fpr": fpr,
        "tpr": tpr,
        "roc_auc": roc_auc,
        "recall": recall_pr,
        "precision": precision_pr,
        "pr_auc": pr_auc,
        "threshold_df": threshold_df,
    }

def calibration_table(y_true_index, disease_prob, positive_index, n_bins=10):
    y = (np.asarray(y_true_index, dtype=int) == int(positive_index)).astype(int)
    p = np.asarray(disease_prob, dtype=float)
    valid = np.isfinite(p)
    y = y[valid]
    p = p[valid]

    rows = []
    edges = np.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if i < n_bins - 1:
            m = (p >= lo) & (p < hi)
        else:
            m = (p >= lo) & (p <= hi)

        if np.sum(m) == 0:
            rows.append([lo, hi, np.nan, np.nan, 0])
        else:
            rows.append([lo, hi, float(np.mean(p[m])), float(np.mean(y[m])), int(np.sum(m))])

    return pd.DataFrame(
        rows,
        columns=["bin_low", "bin_high", "mean_confidence", "fraction_positive", "count"],
    )

def label_is_positive(label):
    s = str(label).lower()
    keywords = ["disease", "pd", "parkinson", "ad", "alz", "mci", "epilepsy", "patient", "positive", "case"]
    return any(k in s for k in keywords)

def label_name_from_index(i, index_to_label):
    return index_to_label.get(int(i), str(i))

required_output_files = {
    "predictions CSV": pred_csv_path,
    "subject summary CSV": subject_csv_path,
    "run summary JSON": run_summary_path,
}

missing_output_files = {
    name: path for name, path in required_output_files.items()
    if not exists(path)
}

if missing_output_files:
    print("\n" + "=" * 100)
    print("PLOTTING CELL STOPPED BEFORE PLOTTING")
    print("=" * 100)
    print("The neural-net output files are missing. Run the neural-net cell first.")
    print("\nMissing files:")
    for name, path in missing_output_files.items():
        print(f"  {name}: {path}")
    raise FileNotFoundError("Neural-net output files are missing. Run the neural-net cell before the plotting cell.")

pred_df = pd.read_csv(pred_csv_path)
subject_df = pd.read_csv(subject_csv_path)
history_df = pd.read_csv(history_csv_path) if exists(history_csv_path) else pd.DataFrame()

run_summary = load_json(run_summary_path)
metadata = load_json(training_metadata_path)

sample_manifest = pd.read_csv(training_manifest_path) if exists(training_manifest_path) else pd.DataFrame()
patient_index_df = pd.read_csv(patient_index_csv) if exists(patient_index_csv) else pd.DataFrame()

modality_registry = []
if exists(modality_registry_path):
    with open(modality_registry_path, "r") as f:
        modality_registry = json.load(f)

label_to_index = metadata.get("label_to_index", {})
index_to_label = {int(v): str(k) for k, v in label_to_index.items()} if label_to_index else {}

positive_class_index = int(run_summary.get("positive_class_index", 1))
positive_class_label = str(run_summary.get("positive_class_label", positive_class_index))
training_enabled = bool(run_summary.get("training_enabled", False))

plot_paths = []

fig = plt.figure(figsize=(13.5, 6.8), facecolor=BLACK)
ax = fig.add_subplot(111)
ax.axis("off")

summary_lines = [
    "Patient-Level NeuroDynamical Neural Network Report",
    "",
    f"Training enabled: {training_enabled}",
    f"Samples / patient-runs: {run_summary.get('n_samples', 'unknown')}",
    f"Model parameters: {run_summary.get('n_params', 'unknown')}",
    f"Classes in data: {run_summary.get('actual_num_classes', 'unknown')}",
    f"Sites in data: {run_summary.get('actual_num_sites', 'unknown')}",
    f"Positive / disease class: {positive_class_label} (index {positive_class_index})",
    f"Original modalities: {run_summary.get('original_modalities', 'unknown')}",
    f"Model modalities used: {run_summary.get('model_modalities', 'unknown')}",
    f"Best model path: {run_summary.get('best_model_path', None)}",
    "",
    "Interpretation note:",
    "These plots are only disease-performance evidence after multiple labeled control and disease patient/runs exist.",
    "If training was skipped or only one class exists, probabilities/logits are architecture sanity checks, not disease evidence.",
]

ax.text(
    0.03,
    0.96,
    "\n".join(summary_lines),
    ha="left",
    va="top",
    color=ACCENT_SOFT,
    fontsize=12,
    family="monospace",
    bbox=dict(facecolor="#101010", edgecolor=ACCENT_DIM, boxstyle="round,pad=0.6", alpha=0.90),
)

plot_paths.append(save_show(fig, "00_model_run_summary_card.png"))

if not history_df.empty and "epoch" in history_df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(13, 8), facecolor=BLACK, sharex=True)

    ax = axes[0]

    if "train_loss" in history_df.columns:
        ax.plot(history_df["epoch"], history_df["train_loss"], color=ACCENT, linewidth=2.0, label="Train loss")

    if "val_loss" in history_df.columns:
        ax.plot(history_df["epoch"], history_df["val_loss"], color=WHITE, linewidth=2.0, label="Validation loss")

    if "val_loss" in history_df.columns and history_df["val_loss"].notna().any():
        best_idx = int(history_df["val_loss"].idxmin())
        best_epoch = float(history_df.loc[best_idx, "epoch"])
        best_loss = float(history_df.loc[best_idx, "val_loss"])

        ax.scatter(
            [best_epoch],
            [best_loss],
            s=90,
            facecolors=BLACK,
            edgecolors=ACCENT_SOFT,
            linewidths=1.5,
            zorder=5,
        )

        ax.annotate(
            f"best val\nepoch {best_epoch:.0f}",
            (best_epoch, best_loss),
            xytext=(8, 8),
            textcoords="offset points",
            color=ACCENT_SOFT,
        )

    style_ax(ax, "Training Loss Dynamics", None, "Loss")

    leg = ax.legend(loc="upper right", frameon=True)
    for t in leg.get_texts():
        t.set_color(ACCENT)

    ax = axes[1]

    if "train_accuracy" in history_df.columns:
        ax.plot(history_df["epoch"], history_df["train_accuracy"], color=ACCENT, linewidth=2.0, label="Train accuracy")

    if "val_accuracy" in history_df.columns:
        ax.plot(history_df["epoch"], history_df["val_accuracy"], color=WHITE, linewidth=2.0, label="Validation accuracy")

    if "lr" in history_df.columns:
        ax2 = ax.twinx()
        ax2.plot(history_df["epoch"], history_df["lr"], color=ACCENT_DIM, linewidth=1.3, linestyle="--", label="Learning rate")
        ax2.set_ylabel("Learning rate", color=ACCENT_DIM)
        ax2.tick_params(axis="y", colors=ACCENT_DIM)

        for spine in ax2.spines.values():
            spine.set_color(ACCENT_DIM)

    style_ax(ax, "Accuracy and Learning-Rate Schedule", "Epoch", "Accuracy")
    ax.set_ylim(-0.05, 1.05)

    leg = ax.legend(loc="lower right", frameon=True)
    for t in leg.get_texts():
        t.set_color(ACCENT)

    plot_paths.append(save_show(fig, "01_training_loss_accuracy_lr.png"))

    if {"train_loss", "val_loss"}.issubset(history_df.columns):
        gap = history_df["val_loss"].to_numpy(dtype=float) - history_df["train_loss"].to_numpy(dtype=float)

        fig, ax = plt.subplots(figsize=(12, 4.8), facecolor=BLACK)

        ax.plot(history_df["epoch"], gap, color=ACCENT, linewidth=2.2)
        ax.axhline(0, color=WHITE, linestyle="--", linewidth=1.0, alpha=0.70)

        late_gap = np.nanmedian(gap[-max(1, min(10, len(gap))):])

        ax.text(
            0.02,
            0.96,
            f"Late median val-train loss gap: {late_gap:.4f}\nPositive rising gap suggests overfit; near-zero gap suggests stable fit.",
            transform=ax.transAxes,
            ha="left",
            va="top",
            color=ACCENT_SOFT,
            fontsize=9,
            bbox=dict(facecolor=BLACK, edgecolor=ACCENT_DIM, alpha=0.75),
        )

        style_ax(ax, "Generalization Gap Diagnostic", "Epoch", "Validation loss - train loss")

        plot_paths.append(save_show(fig, "02_generalization_gap.png"))

y_true = pred_df["label_index"].to_numpy(dtype=int) if "label_index" in pred_df.columns else np.zeros(len(pred_df), dtype=int)
y_pred = pred_df["predicted_class_index"].to_numpy(dtype=int) if "predicted_class_index" in pred_df.columns else np.zeros(len(pred_df), dtype=int)
classes = sorted(set(y_true.tolist() + y_pred.tolist()))

if len(classes) > 0:
    cm = confusion_matrix_np(y_true, y_pred, classes)
    cm_norm = cm / (cm.sum(axis=1, keepdims=True) + eps)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8), facecolor=BLACK)

    ax = axes[0]

    im = ax.imshow(cm, cmap=CYAN_SEQUENTIAL_CMAP, interpolation="nearest")

    ax.set_xticks(np.arange(len(classes)))
    ax.set_xticklabels([label_name_from_index(c, index_to_label) for c in classes], rotation=35, ha="right", color=ACCENT)

    ax.set_yticks(np.arange(len(classes)))
    ax.set_yticklabels([label_name_from_index(c, index_to_label) for c in classes], color=ACCENT)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=WHITE, fontsize=11)

    style_ax(ax, "Confusion Matrix\ncounts", "Predicted", "True", heatmap=True)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    style_cbar(cbar, "Count")

    ax = axes[1]

    im = ax.imshow(cm_norm, cmap=CYAN_SEQUENTIAL_CMAP, interpolation="nearest", vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(classes)))
    ax.set_xticklabels([label_name_from_index(c, index_to_label) for c in classes], rotation=35, ha="right", color=ACCENT)

    ax.set_yticks(np.arange(len(classes)))
    ax.set_yticklabels([label_name_from_index(c, index_to_label) for c in classes], color=ACCENT)

    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color=WHITE, fontsize=10)

    acc = np.trace(cm) / (np.sum(cm) + eps)
    bal_acc = np.mean(np.diag(cm) / (np.sum(cm, axis=1) + eps))

    style_ax(
        ax,
        f"Normalized Confusion Matrix\naccuracy={acc:.3f}, balanced accuracy={bal_acc:.3f}",
        "Predicted",
        "True",
        heatmap=True,
    )

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    style_cbar(cbar, "Row-normalized")

    plot_paths.append(save_show(fig, "03_confusion_matrix_and_balanced_accuracy.png"))

curve_data = None

if "disease_probability" in pred_df.columns:
    disease_prob = pred_df["disease_probability"].to_numpy(dtype=float)
    curve_data = binary_curves(y_true, disease_prob, positive_class_index)

    if curve_data is not None:
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5), facecolor=BLACK)

        ax = axes[0]

        ax.plot(
            curve_data["fpr"],
            curve_data["tpr"],
            color=ACCENT,
            linewidth=2.4,
            label=f"ROC AUC = {curve_data['roc_auc']:.3f}",
        )

        ax.plot([0, 1], [0, 1], color=ACCENT_DIM, linestyle="--", linewidth=1.0, alpha=0.8)

        style_ax(ax, "ROC Curve", "False positive rate", "True positive rate")

        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)

        leg = ax.legend(loc="lower right", frameon=True)
        for t in leg.get_texts():
            t.set_color(ACCENT)

        ax = axes[1]

        ax.plot(
            curve_data["recall"],
            curve_data["precision"],
            color=ACCENT,
            linewidth=2.4,
            label=f"PR AUC = {curve_data['pr_auc']:.3f}",
        )

        base_rate = np.mean(y_true == positive_class_index)
        ax.axhline(base_rate, color=ACCENT_DIM, linestyle="--", linewidth=1.0, alpha=0.8, label=f"Base rate = {base_rate:.3f}")

        style_ax(ax, "Precision-Recall Curve", "Recall", "Precision")

        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)

        leg = ax.legend(loc="lower left", frameon=True)
        for t in leg.get_texts():
            t.set_color(ACCENT)

        plot_paths.append(save_show(fig, "04_roc_precision_recall.png"))

        th_df = curve_data["threshold_df"].sort_values("balanced_accuracy", ascending=False)
        th_df.to_csv(os.path.join(plot_dir, "threshold_sweep_metrics.csv"), index=False)

        fig, ax = plt.subplots(figsize=(12.5, 5.5), facecolor=BLACK)

        ax.plot(th_df["threshold"], th_df["balanced_accuracy"], color=ACCENT, linewidth=2.0, label="Balanced accuracy")
        ax.plot(th_df["threshold"], th_df["sensitivity"], color=WHITE, linewidth=1.6, label="Sensitivity")
        ax.plot(th_df["threshold"], th_df["specificity"], color=ACCENT_SOFT, linewidth=1.6, label="Specificity")
        ax.plot(th_df["threshold"], th_df["f1"], color=ACCENT_DIM, linewidth=1.5, linestyle="--", label="F1")

        best = th_df.iloc[0]

        ax.axvline(best["threshold"], color=WARN, linestyle="--", linewidth=1.2)

        ax.text(
            best["threshold"],
            0.04,
            f"best threshold={best['threshold']:.3f}\nbal acc={best['balanced_accuracy']:.3f}",
            color=WARN,
            fontsize=9,
            ha="left",
            va="bottom",
            bbox=dict(facecolor=BLACK, edgecolor=WARN, alpha=0.75),
        )

        style_ax(ax, "Decision Threshold Sweep", "Disease-probability threshold", "Metric")
        ax.set_ylim(-0.05, 1.05)

        leg = ax.legend(loc="upper right", frameon=True)
        for t in leg.get_texts():
            t.set_color(ACCENT)

        plot_paths.append(save_show(fig, "05_threshold_sweep.png"))

        cal_df = calibration_table(y_true, disease_prob, positive_class_index, n_bins=10)
        cal_df.to_csv(os.path.join(plot_dir, "calibration_table.csv"), index=False)

        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), facecolor=BLACK)

        ax = axes[0]

        ax.plot([0, 1], [0, 1], color=ACCENT_DIM, linestyle="--", linewidth=1.1, alpha=0.85, label="Perfect calibration")

        valid_cal = cal_df[cal_df["count"] > 0]

        ax.plot(
            valid_cal["mean_confidence"],
            valid_cal["fraction_positive"],
            color=ACCENT,
            marker="o",
            linewidth=2.0,
            label="Observed",
        )

        for _, row in valid_cal.iterrows():
            ax.text(
                row["mean_confidence"],
                row["fraction_positive"],
                str(int(row["count"])),
                color=ACCENT_SOFT,
                fontsize=8,
                ha="left",
                va="bottom",
            )

        style_ax(
            ax,
            "Reliability / Calibration Curve\nnumber labels = bin counts",
            "Mean predicted disease probability",
            "Observed positive fraction",
        )

        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)

        leg = ax.legend(loc="upper left", frameon=True)
        for t in leg.get_texts():
            t.set_color(ACCENT)

        ax = axes[1]

        pos = disease_prob[y_true == positive_class_index]
        neg = disease_prob[y_true != positive_class_index]
        bins = np.linspace(0, 1, 16)

        ax.hist(neg, bins=bins, color=ACCENT_DIM, alpha=0.55, edgecolor=ACCENT_DIM, label="Non-disease / other")
        ax.hist(pos, bins=bins, color=ACCENT, alpha=0.55, edgecolor=ACCENT_SOFT, label="Positive / disease")

        ax.axvline(0.5, color=WHITE, linestyle="--", linewidth=1.1, alpha=0.8)

        style_ax(ax, "Disease Probability Distribution by True Class", "Predicted disease probability", "Count")

        leg = ax.legend(loc="upper center", frameon=True)
        for t in leg.get_texts():
            t.set_color(ACCENT)

        plot_paths.append(save_show(fig, "06_calibration_and_probability_distribution.png"))

if not subject_df.empty and "disease_probability_median" in subject_df.columns:
    sdf = subject_df.copy()

    sdf["display_id"] = sdf.get("patient_id", pd.Series(np.arange(len(sdf)))).astype(str)
    sdf["label_str"] = sdf.get("label", pd.Series(["unknown"] * len(sdf))).astype(str)
    sdf["is_positive_label"] = sdf["label_str"].map(label_is_positive)

    sdf = sdf.sort_values("disease_probability_median", ascending=True).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12.5, max(5.8, 0.36 * len(sdf))), facecolor=BLACK)

    y = np.arange(len(sdf))
    colors = [ACCENT if x else ACCENT_DIM for x in sdf["is_positive_label"]]

    ax.barh(y, sdf["disease_probability_median"], color=colors, edgecolor=ACCENT_SOFT, alpha=0.86)
    ax.axvline(0.5, color=WHITE, linestyle="--", linewidth=1.1, alpha=0.8, label="0.5 threshold")

    ax.set_yticks(y)
    ax.set_yticklabels(sdf["display_id"], color=ACCENT)

    for yi, (_, row) in zip(y, sdf.iterrows()):
        ax.text(
            row["disease_probability_median"],
            yi,
            f"  {row['label_str']} | med={row['disease_probability_median']:.3f}",
            va="center",
            ha="left",
            color=ACCENT_SOFT,
            fontsize=8,
        )

    style_ax(ax, "Patient-Level Disease Probability Ranking", "Median disease probability", "Patient / run")
    ax.set_xlim(0, 1.05)

    leg = ax.legend(loc="lower right", frameon=True)
    for t in leg.get_texts():
        t.set_color(ACCENT)

    plot_paths.append(save_show(fig, "07_patient_level_disease_probability_ranking.png"))

    if "uncertainty_mean" in sdf.columns:
        fig, ax = plt.subplots(figsize=(9.5, 6.5), facecolor=BLACK)

        x = sdf["disease_probability_median"].to_numpy(dtype=float)
        yv = sdf["uncertainty_mean"].to_numpy(dtype=float)

        ax.scatter(x, yv, s=140, color=ACCENT, edgecolors=WHITE, linewidths=0.8, alpha=0.85)
        ax.axvline(0.5, color=WHITE, linestyle="--", linewidth=1.0, alpha=0.70)
        ax.axhline(np.nanmedian(yv), color=ACCENT_DIM, linestyle="--", linewidth=1.0, alpha=0.70)

        for _, row in sdf.iterrows():
            ax.annotate(
                row["display_id"],
                (row["disease_probability_median"], row["uncertainty_mean"]),
                xytext=(5, 5),
                textcoords="offset points",
                color=ACCENT_SOFT,
                fontsize=8,
            )

        style_ax(
            ax,
            "Probability vs Uncertainty\nhigh-probability + low-uncertainty cases are most stable",
            "Median disease probability",
            "Mean uncertainty score",
        )

        plot_paths.append(save_show(fig, "08_probability_uncertainty_quadrant.png"))

embeddings = to_numpy(load_torch(embeddings_path)) if exists(embeddings_path) else None

if embeddings is not None and len(embeddings) == len(pred_df):
    coords, explained = pca_2d(embeddings)

    fig, ax = plt.subplots(figsize=(9.5, 7), facecolor=BLACK)

    if "disease_probability" in pred_df.columns:
        cvals = pred_df["disease_probability"].to_numpy(dtype=float)
    else:
        cvals = y_true.astype(float)

    sc = ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=cvals,
        s=120,
        cmap=CYAN_SEQUENTIAL_CMAP,
        edgecolors=WHITE,
        linewidths=0.7,
        alpha=0.88,
    )

    if "patient_id" in pred_df.columns:
        for i, row in pred_df.iterrows():
            ax.annotate(
                str(row["patient_id"]),
                (coords[i, 0], coords[i, 1]),
                xytext=(5, 5),
                textcoords="offset points",
                color=ACCENT_SOFT,
                fontsize=8,
            )

    style_ax(
        ax,
        f"Embedding PCA Projection\nPC1={explained[0]*100:.1f}%, PC2={explained[1]*100:.1f}%",
        "Embedding PC1",
        "Embedding PC2",
    )

    cbar = plt.colorbar(sc, ax=ax)
    style_cbar(cbar, "Disease probability")

    plot_paths.append(save_show(fig, "09_embedding_pca_disease_probability.png"))

    emb_df = pd.DataFrame(coords, columns=["embedding_pc1", "embedding_pc2"])
    emb_df.insert(0, "sample_index", np.arange(len(emb_df)))
    emb_df["label_index"] = y_true

    if "disease_probability" in pred_df.columns:
        emb_df["disease_probability"] = pred_df["disease_probability"].to_numpy(dtype=float)

    emb_df.to_csv(os.path.join(plot_dir, "embedding_pca_coordinates.csv"), index=False)

logits_np = to_numpy(load_torch(logits_path)) if exists(logits_path) else None

if logits_np is not None and logits_np.ndim == 2 and logits_np.shape[0] == len(pred_df):
    logits_np = logits_np.astype(float)
    logits_np = logits_np - np.nanmax(logits_np, axis=1, keepdims=True)
    probs_np = np.exp(logits_np) / (np.sum(np.exp(logits_np), axis=1, keepdims=True) + eps)

    pred_entropy = -np.sum(probs_np * np.log(probs_np + eps), axis=1)
    sorted_probs = np.sort(probs_np, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2] if sorted_probs.shape[1] >= 2 else sorted_probs[:, -1]

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.3), facecolor=BLACK)

    ax = axes[0]

    ax.hist(margin, bins=16, color=ACCENT, alpha=0.70, edgecolor=ACCENT_SOFT)
    ax.axvline(np.nanmedian(margin), color=WHITE, linestyle="--", linewidth=1.1, alpha=0.80, label=f"Median={np.nanmedian(margin):.3f}")

    style_ax(
        ax,
        "Prediction Margin Distribution\nlarger margin = more decisive softmax output",
        "Top probability - second probability",
        "Count",
    )

    leg = ax.legend(loc="upper right", frameon=True)
    for t in leg.get_texts():
        t.set_color(ACCENT)

    ax = axes[1]

    ax.hist(pred_entropy, bins=16, color=ACCENT_DIM, alpha=0.70, edgecolor=ACCENT)
    ax.axvline(np.nanmedian(pred_entropy), color=WHITE, linestyle="--", linewidth=1.1, alpha=0.80, label=f"Median={np.nanmedian(pred_entropy):.3f}")

    style_ax(
        ax,
        "Predictive Entropy Distribution\nhigher entropy = less certain class distribution",
        "Softmax entropy",
        "Count",
    )

    leg = ax.legend(loc="upper right", frameon=True)
    for t in leg.get_texts():
        t.set_color(ACCENT)

    plot_paths.append(save_show(fig, "10_margin_and_entropy_diagnostics.png"))

bundle_obj = load_torch(training_bundle_path) if exists(training_bundle_path) else None

if bundle_obj is not None and isinstance(bundle_obj, dict) and "dynamical_mask" in bundle_obj:
    mask_np = to_numpy(bundle_obj["dynamical_mask"]).astype(float)
    tokens_np = to_numpy(bundle_obj["dynamical_tokens"]) if "dynamical_tokens" in bundle_obj else None
    images_np = to_numpy(bundle_obj["dynamical_image_bank"]) if "dynamical_image_bank" in bundle_obj else None

    if mask_np is not None and mask_np.ndim == 2:
        m = mask_np.astype(float)
        n_show_modalities = min(80, m.shape[1])
        coverage = np.nanmean(m, axis=0)

        if len(modality_registry) == m.shape[1]:
            modality_names = modality_registry
        else:
            modality_names = [f"modality_{i:03d}" for i in range(m.shape[1])]

        order = np.argsort(-coverage)
        show_idx = order[:n_show_modalities]
        show_mask = m[:, show_idx]
        show_names = [modality_names[i] for i in show_idx]

        fig, ax = plt.subplots(figsize=(13.5, max(4.5, 0.28 * m.shape[0])), facecolor=BLACK)

        im = ax.imshow(show_mask, aspect="auto", interpolation="nearest", cmap=CYAN_SEQUENTIAL_CMAP, vmin=0, vmax=1)

        ax.set_yticks(np.arange(m.shape[0]))

        if not sample_manifest.empty and "patient_id" in sample_manifest.columns:
            ax.set_yticklabels(sample_manifest["patient_id"].astype(str).tolist(), color=ACCENT, fontsize=8)
        else:
            ax.set_yticklabels([f"sample_{i}" for i in range(m.shape[0])], color=ACCENT, fontsize=8)

        ax.set_xticks(np.arange(n_show_modalities))
        ax.set_xticklabels([s[:28] for s in show_names], rotation=80, ha="right", color=ACCENT, fontsize=7)

        style_ax(
            ax,
            f"Modality Coverage Matrix\ntop {n_show_modalities} modalities by presence",
            "Dynamical result modality",
            "Patient / run",
            heatmap=True,
        )

        cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
        style_cbar(cbar, "Present")

        plot_paths.append(save_show(fig, "11_modality_coverage_matrix.png"))

        top_n = min(40, len(order))
        top_idx = order[:top_n][::-1]

        fig, ax = plt.subplots(figsize=(12, max(5.5, 0.25 * top_n)), facecolor=BLACK)

        y = np.arange(top_n)

        ax.barh(y, coverage[top_idx], color=ACCENT, edgecolor=ACCENT_SOFT, alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels([modality_names[i][:50] for i in top_idx], color=ACCENT, fontsize=8)

        style_ax(
            ax,
            "Most Consistently Available Dynamical Modalities",
            "Fraction of patient/runs with modality present",
            "Modality",
        )

        ax.set_xlim(0, 1.05)

        plot_paths.append(save_show(fig, "12_top_modality_coverage_ranking.png"))

        coverage_df = pd.DataFrame({
            "modality_index": np.arange(len(coverage)),
            "modality_name": modality_names,
            "coverage_fraction": coverage,
        }).sort_values("coverage_fraction", ascending=False)

        coverage_df.to_csv(os.path.join(plot_dir, "modality_coverage_table.csv"), index=False)

    if tokens_np is not None and mask_np is not None and tokens_np.ndim == 3:
        token_strength = np.nanmean(np.abs(tokens_np), axis=2)
        token_strength = np.where(mask_np > 0, token_strength, np.nan)

        mean_strength = np.nanmean(token_strength, axis=0)

        if len(modality_registry) == len(mean_strength):
            modality_names = modality_registry
        else:
            modality_names = [f"modality_{i:03d}" for i in range(len(mean_strength))]

        strength_df = pd.DataFrame({
            "modality_index": np.arange(len(mean_strength)),
            "modality_name": modality_names,
            "mean_token_strength": mean_strength,
            "coverage_fraction": np.nanmean(mask_np, axis=0),
        }).sort_values("mean_token_strength", ascending=False)

        strength_df.to_csv(os.path.join(plot_dir, "modality_token_strength_table.csv"), index=False)

        top = strength_df.head(min(35, len(strength_df))).iloc[::-1]

        fig, ax = plt.subplots(figsize=(12, max(5.8, 0.30 * len(top))), facecolor=BLACK)

        y = np.arange(len(top))

        ax.barh(y, top["mean_token_strength"], color=ACCENT, edgecolor=ACCENT_SOFT, alpha=0.86)
        ax.set_yticks(y)
        ax.set_yticklabels(top["modality_name"].astype(str).str[:55], color=ACCENT, fontsize=8)

        style_ax(
            ax,
            "Strongest Dynamical Token Modalities",
            "Mean absolute token strength",
            "Modality",
        )

        plot_paths.append(save_show(fig, "13_modality_token_strength.png"))

if not patient_index_df.empty and "n_modalities_loaded" in patient_index_df.columns:
    pidx = patient_index_df.copy()
    pidx["display_id"] = pidx["patient_id"].astype(str) + " / " + pidx["run_id"].astype(str)
    pidx = pidx.sort_values("n_modalities_loaded", ascending=True)

    fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.35 * len(pidx))), facecolor=BLACK)

    y = np.arange(len(pidx))

    ax.barh(y, pidx["n_modalities_loaded"], color=ACCENT, edgecolor=ACCENT_SOFT, alpha=0.86)

    ax.set_yticks(y)
    ax.set_yticklabels(pidx["display_id"], color=ACCENT, fontsize=8)

    for yi, (_, row) in zip(y, pidx.iterrows()):
        ax.text(
            row["n_modalities_loaded"],
            yi,
            f"  {row['label']}",
            color=ACCENT_SOFT,
            va="center",
            fontsize=8,
        )

    style_ax(
        ax,
        "Per-Patient Dynamical Output Count\nlow counts may indicate incomplete notebook runs",
        "Loaded modalities",
        "Patient / run",
    )

    plot_paths.append(save_show(fig, "14_patient_output_completeness.png"))

summary_rows = []

summary_rows.append(["training_enabled", training_enabled])
summary_rows.append(["n_samples", run_summary.get("n_samples", np.nan)])
summary_rows.append(["n_params", run_summary.get("n_params", np.nan)])
summary_rows.append(["actual_num_classes", run_summary.get("actual_num_classes", np.nan)])
summary_rows.append(["actual_num_sites", run_summary.get("actual_num_sites", np.nan)])
summary_rows.append(["original_modalities", run_summary.get("original_modalities", np.nan)])
summary_rows.append(["model_modalities", run_summary.get("model_modalities", np.nan)])

if "disease_probability" in pred_df.columns:
    summary_rows.append(["sample_probability_mean", float(np.nanmean(pred_df["disease_probability"]))])
    summary_rows.append(["sample_probability_median", float(np.nanmedian(pred_df["disease_probability"]))])
    summary_rows.append(["sample_probability_std", float(np.nanstd(pred_df["disease_probability"]))])

if not subject_df.empty and "disease_probability_median" in subject_df.columns:
    summary_rows.append(["subject_probability_median_mean", float(np.nanmean(subject_df["disease_probability_median"]))])
    summary_rows.append(["subject_probability_median_std", float(np.nanstd(subject_df["disease_probability_median"]))])

if len(classes) > 0:
    summary_rows.append(["sample_accuracy", float(np.trace(cm) / (np.sum(cm) + eps))])
    summary_rows.append(["sample_balanced_accuracy", float(np.mean(np.diag(cm) / (np.sum(cm, axis=1) + eps)))])

if curve_data is not None:
    summary_rows.append(["roc_auc", float(curve_data["roc_auc"])])
    summary_rows.append(["pr_auc", float(curve_data["pr_auc"])])

summary_df = pd.DataFrame(summary_rows, columns=["metric", "value"])
summary_metrics_path = os.path.join(plot_dir, "expert_plot_summary_metrics.csv")
summary_df.to_csv(summary_metrics_path, index=False)

plot_manifest = pd.DataFrame({
    "plot_path": plot_paths,
    "plot_file": [os.path.basename(p) for p in plot_paths],
})

plot_manifest_path = os.path.join(plot_dir, "expert_plot_manifest.csv")
plot_manifest.to_csv(plot_manifest_path, index=False)

print("\n" + "=" * 100)
print("EXPERT NEURAL-NET PERFORMANCE PLOTS COMPLETE")
print("=" * 100)
print(f"Plot folder: {plot_dir}")
print(f"Plot manifest: {plot_manifest_path}")
print(f"Summary metrics: {summary_metrics_path}")

print("\nGenerated plots:")
for p in plot_paths:
    print(p)
    
print("\nSummary metrics:")
print(summary_df.to_string(index=False))